# CodeBlue | CMEMS monthly + WOA23 — regional visualization

Reusable analysis of physical/biogeochemical boundary data for CodeBlue. **The area downloaded and the area visualized need not be identical.** Downloaders remain unchanged.

1. Run **Section 1** and select whether the visualization should use the full downloaded area (**Yes**) or the boundary geometry of a **CROCO NetCDF grid** (**No**). In the second case provide a server-accessible `.nc` path or upload a `.nc` file. No manual rotation is needed.
2. Choose the active/open boundaries and band half-width (default **±1° latitude/longitude**; optional km for geodesically more meaningful comparisons).
3. Run **Section 2** to load the data. The downloaded area is detected from the CMEMS files. Click **Apply geometry** after changing settings, then run the map (Section 3) to verify the geometry **before** interpreting the diagnostics.

**Interpretation:** In the geographic overview, a **black solid line** shows the visualization/model boundary and a **gray dashed line** shows the actual downloaded-data envelope. Colored areas are the boundary sampling bands, which may overlap near corners. They are not separate model domains.

The NetCDF must have matching 2-D coordinates such as `lon_rho`, `lat_rho`; the outermost rho-grid rows/columns are treated as four edges. Check for halo cells and select only truly open boundaries.

**Boundary selection:** In Section 1, tick the open boundaries to analyse. The default for the CodeBlue MANGAE2500 example is North, South and West (East excluded). All boundary-dependent plots and statistics only use the selected sides. After changing the selection, click **Apply geometry** and re-run the subsequent analysis cells.


**Model vs download limits:** In Section 1 the *manual* values describe the original study domain, not the padded download rectangle. Select **Manual model bounds** or **Read model bounds from .nc**. For MANGAE2500 use (−18, 9.5, 41, 55) in degrees; when download padding is ±1° the data should cover approximately (−19, 10.5, 40, 56).


**Mediterranean Sea exclusion:** Enabled by default; applied uniformly to CMEMS, WOA23 monthly and WOA23 annual fields before visualization/statistics. See Section 2 for its approximate regional polygon.


## 1 — Define the model domain and select active boundaries

**Two options:** enter the four *original study-domain coordinates* manually (without the download padding), or load the actual perimeter from a CROCO NetCDF grid. The downloaded CMEMS/WOA23 coverage is determined independently from the source files.

For the CodeBlue MANGAE2500 example, manually enter **W −18°, E 9.5°, S 41°, N 55°**, then select North, South and West. With ±1° sampling bands, the required download extent is approximately **W −19°, E 10.5°, S 40°, N 56°**.


**Mediterranean Sea:** The *Exclude Mediterranean Sea data* checkbox is enabled by default. The option is applied to the loaded CMEMS and WOA23 fields at the beginning of Section 2, before generating any maps or boundary statistics. It does not edit the downloaded files.


In [ ]:
from pathlib import Path
from io import BytesIO
import re, warnings, math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from dask import compute
from shapely.geometry import LineString, Polygon, box
from shapely.ops import transform as shapely_transform
from shapely import contains_xy, intersects_xy
from pyproj import CRS, Transformer

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'download'/'download_cmems_bgc.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run Jupyter inside the codeblue-obc-ini repository.')
CMEMS_DIR = ROOT/'data'/('CMEMS_GRID' if (ROOT/'data'/'CMEMS_GRID').exists() else 'CMEMS')
WOA_DIR = ROOT/'data'/('WOA_GRID' if (ROOT/'data'/'WOA_GRID').exists() else 'WOA')
VARS = {'no3':'Nitrate', 'po4':'Phosphate', 'si':'Silicate', 'o2':'Dissolved oxygen'}
UNITS = 'mmol m$^{-3}$'
# Updated automatically in Section 2 from actual CMEMS geographic coordinates.
DOWNLOAD_BOUNDS = None  # west, east, south, north
MAX_MONTHS = None  # None = all completed CMEMS files available
HOV_VARIABLE = 'no3'
HOV_MAX_DEPTH = 800.0
plt.rcParams.update({'figure.dpi': 110})

# Geometry state; all analysis cells use REGIONS from this configuration.
REGIONS, BOUNDARY_LINES, BOUNDARY_POLYGONS = {}, {}, {}
MODEL_OUTLINE = None
MODEL_BOUNDS = None
GEOMETRY_MODE = 'rectangle'
BOUNDARY_COLORS = {}


# Small, visible instructions attached to each interactive input.
# All guidance is in English so collaborators can use this notebook.
def guided_control(control, hint):
    return widgets.VBox(
        [control, widgets.HTML(
            value=f'<div style="max-width:235px;color:#526477;font-size:12px;line-height:1.4;">{hint}</div>'
        )],
        layout=widgets.Layout(margin='0 12px 10px 0', align_items='flex-start')
    )

# Download bounds and model/visualization bounds are DIFFERENT concepts.
# The manual option below specifies ORIGINAL model bounds, not padded download bounds.
same_limits = widgets.ToggleButtons(
    options=[
        ('Manual model bounds', 'manual'),
        ('Read model bounds from .nc', 'netcdf')
    ],
    value='manual',
    description='Domain definition:',
    style={'description_width': 'initial'}
)

manual_bounds = {
    'west': widgets.FloatText(value=-18.0, description='West (°):',
                              style={'description_width': 'initial'},
                              layout=widgets.Layout(width='170px')),
    'east': widgets.FloatText(value=9.5, description='East (°):',
                              style={'description_width': 'initial'},
                              layout=widgets.Layout(width='170px')),
    'south': widgets.FloatText(value=41.0, description='South (°):',
                               style={'description_width': 'initial'},
                               layout=widgets.Layout(width='170px')),
    'north': widgets.FloatText(value=55.0, description='North (°):',
                               style={'description_width': 'initial'},
                               layout=widgets.Layout(width='170px')),
}
manual_panel = widgets.VBox([
    widgets.HTML(
        '<b>Original model limits (NOT download limits)</b><br>'
        '<span style="font-size:12px;color:#526477;">'
        'Enter the four sides of your study domain. Do not include '
        'the additional download padding. Example: MANGAE2500 = '
        'W −18°, E 9.5°, S 41°, N 55°.</span>'
    ),
    widgets.HBox(list(manual_bounds.values()),
                 layout=widgets.Layout(flex_flow='row wrap'))
])
# Exclude the Mediterranean basin before ANY boundary-dependent calculations.
# This mask removes Mediterranean grid cells from the in-memory CMEMS/WOA datasets,
# without changing downloaded files on disk.
mediterranean_checkbox = widgets.Checkbox(
    value=True,
    description='Exclude Mediterranean Sea data',
    indent=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='310px')
)
mediterranean_setting = widgets.VBox([
    mediterranean_checkbox,
    widgets.HTML(
        '<div style="font-size:12px;color:#526477;max-width:700px;">'
        'Recommended for the NE Atlantic CodeBlue analyses. '
        'Mediterranean cells are set to NaN in CMEMS and WOA23 (monthly and annual). '
        'The Strait of Gibraltar separates the basins. '
        'The exclusion is based on a documented regional polygon, not on the '
        'longitude alone, so the Bay of Biscay remains included. '
        'After changing this option, rerun Section 2 and subsequent analyses.'
        '</div>'
    )
])

netcdf_path = widgets.Text(
    value='', placeholder='/path/to/croco_grd.nc', description='Grid (.nc) path:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='610px')
)
netcdf_upload = widgets.FileUpload(accept='.nc', multiple=False, description='Upload .nc')
band_units = widgets.Dropdown(
    options=[('Degrees of lon/lat (±°)', 'degrees'), ('Distance (±km)', 'km')],
    value='degrees', description='Band units:', style={'description_width': 'initial'}
)
band_half_width = widgets.BoundedFloatText(
    value=1.0, min=0.01, max=500, step=0.1,
    description='Band half-width:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
# Checkboxes make it clear that any combination of boundaries is allowed.
# Disabled boundaries will be omitted from every analysis and figure.
boundary_checkboxes = {
    name: widgets.Checkbox(
        value=(name != 'East'),  # CodeBlue/MANGAE2500 default: East excluded
        description=name,
        indent=False,
        layout=widgets.Layout(width='110px')
    )
    for name in ('North', 'South', 'West', 'East')
}
boundary_selector = widgets.VBox([
    widgets.HTML(
        '<b>Which boundaries do you want to analyse?</b><br>'
        '<span style="color:#526477;font-size:12px;">'
        'Tick only the open boundaries of your model. '
        'Unticked boundaries will not be plotted or included in statistics.'
        '</span>'
    ),
    widgets.HBox(
        list(boundary_checkboxes.values()),
        layout=widgets.Layout(flex_flow='row wrap')
    ),
])
apply_button = widgets.Button(description='Apply geometry', button_style='success', icon='check')
geometry_message = widgets.HTML()
custom_panel = widgets.VBox([widgets.HBox([netcdf_path, netcdf_upload])])

def toggle_geometry(change=None):
    manual_panel.layout.display = '' if same_limits.value == 'manual' else 'none'
    custom_panel.layout.display = '' if same_limits.value == 'netcdf' else 'none'

same_limits.observe(toggle_geometry, names='value')
toggle_geometry()

def local_transformers(lons,lats):
    lon0,lat0=float(np.nanmean(lons)),float(np.nanmean(lats))
    proj=CRS.from_proj4(f'+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs')
    return (Transformer.from_crs('EPSG:4326',proj,always_xy=True),
            Transformer.from_crs(proj,'EPSG:4326',always_xy=True))

def rectangle_geometry(w,e,s,n,band,active):
    if not (-180<=w<e<=180 and -90<=s<n<=90 and band>0):
        raise ValueError('Check rectangle bounds and positive band width.')
    outline=Polygon([(w,s),(e,s),(e,n),(w,n)])
    lines={'South':LineString([(w,s),(e,s)]), 'East':LineString([(e,s),(e,n)]),
           'North':LineString([(w,n),(e,n)]), 'West':LineString([(w,s),(w,n)])}
    # Exact longitude/latitude degree bands, extended at BOTH endpoints.
    shapes={'South':box(w-band,s-band,e+band,s+band),
            'North':box(w-band,n-band,e+band,n+band),
            'West':box(w-band,s-band,w+band,n+band),
            'East':box(e-band,s-band,e+band,n+band)}
    return outline,{k:lines[k] for k in active},{k:shapes[k] for k in active}

def boundary_polygons(outline, lines, width, units, active):
    """Return actual buffer polygons around edges, retaining their real shapes.

    Degrees: geographic-coordinate buffer (not a constant physical width).
    Kilometres: local azimuthal-equidistant projected buffer.
    """
    if not np.isfinite(width) or width <= 0:
        raise ValueError('Band half-width must be positive and finite.')
    polygons = {}
    if units == 'degrees':
        for name in active:
            polygons[name] = lines[name].buffer(width, cap_style=3, join_style=2)
    elif units == 'km':
        lons, lats = outline.exterior.xy
        fw, bw = local_transformers(lons, lats)
        for name in active:
            line_metric = shapely_transform(fw.transform, lines[name])
            strip_metric = line_metric.buffer(width * 1000, cap_style=3, join_style=2)
            polygons[name] = shapely_transform(bw.transform, strip_metric)
    else:
        raise ValueError(f'Unknown units: {units}')
    return polygons

def grid_from_netcdf():
    content=None
    if netcdf_upload.value:
        v=netcdf_upload.value
        item=(next(iter(v.values())) if isinstance(v,dict) else v[0])
        content=bytes(item['content'])
        # Save into ignored data/ (never into tracked visualization source)
        target=ROOT/'data'/'uploaded_domain_grid.nc'
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(content)
        path=target
    else:
        raw_path=Path(netcdf_path.value.strip()).expanduser()
        path=raw_path if raw_path.is_absolute() else ROOT/raw_path
    if not path.is_file():raise FileNotFoundError(f'Grid NetCDF not found: {path}')
    with xr.open_dataset(path) as g:
        pairs=[('lon_rho','lat_rho'),('lon','lat'),('longitude','latitude'),('nav_lon','nav_lat')]
        chosen=next(((a,b) for a,b in pairs if a in g and b in g),None)
        if chosen is None:raise ValueError('Expected lon_rho/lat_rho, lon/lat, longitude/latitude or nav_lon/nav_lat.')
        lon=np.asarray(g[chosen[0]].values,dtype=float)
        lat=np.asarray(g[chosen[1]].values,dtype=float)
    if lon.ndim!=2 or lon.shape!=lat.shape or min(lon.shape)<2:
        raise ValueError('NetCDF must contain matching 2-D model-grid longitude/latitude arrays.')
    if not np.isfinite(lon).all() or not np.isfinite(lat).all():
        raise ValueError('Grid edge coordinates must be finite; inspect the input NetCDF.')
    # The rho-grid outer rows/columns are used here; edit edge indices if your grid has halos.
    edges={
        'South':np.column_stack([lon[0,:],lat[0,:]]),
        'East':np.column_stack([lon[:,-1],lat[:,-1]]),
        'North':np.column_stack([lon[-1,::-1],lat[-1,::-1]]),
        'West':np.column_stack([lon[::-1,0],lat[::-1,0]])}
    perimeter=np.concatenate([edges[k][:-1] for k in ('South','East','North','West')])
    outline=Polygon(perimeter)
    if not outline.is_valid or outline.is_empty:
        raise ValueError('Grid outline is invalid/self-intersecting; inspect the grid and halo cells.')
    return outline,{k:LineString(v) for k,v in edges.items()}

def apply_geometry(_=None):
    global REGIONS, BOUNDARY_LINES, BOUNDARY_POLYGONS
    global MODEL_OUTLINE, MODEL_BOUNDS, GEOMETRY_MODE, BOUNDARY_COLORS
    try:
        active = tuple(name for name, checkbox in boundary_checkboxes.items() if checkbox.value)
        if not active:
            raise ValueError('Select at least one active/open model edge.')
        width = float(band_half_width.value)
        units = band_units.value

        if same_limits.value == 'manual':
            w, e, s, n = (float(manual_bounds[key].value)
                          for key in ('west', 'east', 'south', 'north'))
            if not (-180 <= w < e <= 180 and -90 <= s < n <= 90):
                raise ValueError('Invalid original model bounds: require West < East and South < North.')
            outline = box(w, s, e, n)
            lines = {
                'South': LineString([(w,s), (e,s)]),
                'East': LineString([(e,s), (e,n)]),
                'North': LineString([(e,n), (w,n)]),
                'West': LineString([(w,n), (w,s)])
            }
            mode = 'Manual model rectangle'
        else:
            outline, lines = grid_from_netcdf()
            mode = 'CROCO NetCDF grid'

        lines = {k: lines[k] for k in active}
        # Degrees preserves ±1 degree coordinate strips (with square-ended corners);
        # kilometres gives a physically scaled buffer for curved/rotated domains.
        polygons = boundary_polygons(outline, lines, width, units, active)
        xmin, ymin, xmax, ymax = outline.bounds

        if DOWNLOAD_BOUNDS is not None:
            dw, de, ds, dn = DOWNLOAD_BOUNDS
            boxes = [p.bounds for p in polygons.values()]
            if (min(b[0] for b in boxes) < dw - 0.01 or
                max(b[2] for b in boxes) > de + 0.01 or
                min(b[1] for b in boxes) < ds - 0.01 or
                max(b[3] for b in boxes) > dn + 0.01):
                warnings.warn('Sampling bands extend beyond downloaded CMEMS coverage. '
                              'Missing locations are not extrapolated.')

        MODEL_OUTLINE, BOUNDARY_LINES, BOUNDARY_POLYGONS = outline, lines, polygons
        REGIONS = dict(polygons)
        MODEL_BOUNDS = (xmin, xmax, ymin, ymax)
        GEOMETRY_MODE = mode
        BOUNDARY_COLORS = {k: plt.get_cmap('tab10')(i) for i, k in enumerate(REGIONS)}
        if '_BOUNDARY_MASK_CACHE' in globals():
            _BOUNDARY_MASK_CACHE.clear()
        geometry_message.value = (
            f'<b>Applied:</b> {mode}; included boundaries: {", ".join(REGIONS)}; '
            f'lon {xmin:.2f}…{xmax:.2f}, lat {ymin:.2f}…{ymax:.2f}. '
            f'Band half-width: ±{width:g} {units}. '
            'Re-run Sections 3 onward after changes.'
        )
    except Exception as ex:
        geometry_message.value = f'<b style="color:#b00">Geometry error:</b> {ex}'
        raise

apply_button.on_click(apply_geometry)
display(
    widgets.HTML(
        '<h3>Model / visualization geometry</h3>'
        '<p><b>Manual model bounds:</b> enter the original study-domain '
        'limits (without padding). <b>NetCDF grid:</b> use the true '
        'perimeter from <code>croco_grd.nc</code>. The downloaded '
        'CMEMS/WOA23 geographic coverage is read independently in Section 2.</p>'
    ),
    guided_control(same_limits, 'Choose how the boundaries of your MODEL are defined, not where the downloaded data end.'),
    manual_panel,
    guided_control(custom_panel, 'For NetCDF mode, enter the full path to croco_grd.nc or upload a .nc file.'),
    widgets.HBox([
        guided_control(band_units, 'Coordinates (degrees) or physical distance (km) around the boundary lines.'),
        guided_control(band_half_width, 'Band extends this far on BOTH sides of each selected boundary.')
    ], layout=widgets.Layout(flex_flow='row wrap')),
    boundary_selector,
    mediterranean_setting,
    widgets.HTML(
        '<div style="color:#526477;font-size:12px;margin:3px 0 8px;">'
        'Click <b>Apply geometry</b> after changing limits or selected boundaries. '
        'Run Section 2 to load the data, then Section 3 to check whether '
        'the downloaded grids cover the bands. Re-run analysis cells after changes.'
        '</div>'
    ),
    apply_button, geometry_message
)
apply_geometry()
print('Local data paths:',CMEMS_DIR,WOA_DIR)


## 2 — Load CMEMS monthly files and WOA23 monthly + annual climatologies

If **Exclude Mediterranean Sea data** is checked in Section 1, a basin-exclusion polygon is applied to CMEMS, WOA23 monthly and WOA23 annual datasets immediately after loading. The mask keeps the Bay of Biscay and Atlantic waters west of the Strait of Gibraltar. This is an approximate NE Atlantic regional classification, not a high-resolution coastline/basin mask; review the map before use in other domains.


In [ ]:
# CMEMS validation: read metadata only, skip current incomplete downloads
pattern = re.compile(r'^BGC_monthly_(\d{4})(\d{2})\.nc$')
validated = []
for p in sorted(CMEMS_DIR.glob('*/BGC_monthly_*.nc')):
    m = pattern.match(p.name)
    if not m or not p.stat().st_size:
        continue
    yy, mm = map(int, m.groups())
    try:
        with xr.open_dataset(p) as check:
            good = (set(VARS).issubset(check.data_vars)
                    and check.sizes.get('time') == 1
                    and int(check.time.dt.year.values[0]) == yy
                    and int(check.time.dt.month.values[0]) == mm)
        if good:
            validated.append(p)
    except (OSError, ValueError, KeyError, IndexError) as ex:
        warnings.warn(f'Skip {p.name}: {ex}')
if not validated:
    raise FileNotFoundError(f'No valid CMEMS monthly NetCDF in {CMEMS_DIR}')
selected = validated if MAX_MONTHS is None else validated[:MAX_MONTHS]
if 'ds_monthly' in globals():
    ds_monthly.close()
ds_monthly = xr.open_mfdataset([str(p) for p in selected], combine='by_coords',
                              chunks={'time': 1, 'depth': 15}, parallel=False,
                              data_vars='minimal', coords='minimal', compat='override').sortby('time')

# WOA23 raw variable names typically n_an/p_an/i_an/o_an, but support prepared files too.
ALIASES = {
    'no3': ('no3', 'n_an', 'nitrate', 'nitrate_an'),
    'po4': ('po4', 'p_an', 'phosphate', 'phosphate_an'),
    'si':  ('si', 'i_an', 'silicate', 'silicate_an'),
    'o2':  ('o2', 'o_an', 'oxygen', 'oxygen_an'),
}
COORD_ALIASES = {
    'lat': 'latitude', 'lon': 'longitude', 'deptht': 'depth',
    'latitude': 'latitude', 'longitude': 'longitude', 'depth': 'depth',
}
def normalize_woa(source, label):
    source = source.copy()
    ren = {key: val for key, val in COORD_ALIASES.items()
           if key in source.dims or key in source.coords if key != val}
    source = source.rename(ren)
    for c in ('latitude', 'longitude', 'depth'):
        if c not in source.coords:
            raise ValueError(f'{label}: missing coordinate {c}. Found {list(source.coords)}')
    found = {}
    for target, options in ALIASES.items():
        candidate = next((k for k in options if k in source.data_vars), None)
        if candidate is None:
            warnings.warn(f'{label}: no {target} variable in {list(source.data_vars)}; filling with NaN')
        else:
            found[target] = candidate
    out = xr.Dataset({k: source[v].squeeze(drop=True) for k, v in found.items()})
    for c in ('latitude', 'longitude', 'depth'):
        out = out.sortby(c)
    # Geographic crossing not expected for this Atlantic domain; normalize -180..180.
    lon = ((out.longitude + 180) % 360) - 180
    out = out.assign_coords(longitude=lon).sortby('longitude')
    return out

woa_by_month = {}
for month in range(1, 13):
    p = WOA_DIR / f'WOA_{month:02d}.nc'
    if p.is_file():
        with xr.open_dataset(p) as raw:
            woa_by_month[month] = normalize_woa(raw.load(), p.name)
    else:
        warnings.warn(f'Missing {p}: monthly WOA23 month {month} unavailable')
annual_file = WOA_DIR / 'WOA_ANNUAL.nc'
woa_annual = None
if annual_file.is_file():
    with xr.open_dataset(annual_file) as raw:
        woa_annual = normalize_woa(raw.load(), annual_file.name)
else:
    warnings.warn(f'Missing WOA annual: {annual_file}')
if not woa_by_month:
    raise FileNotFoundError(f'No WOA_01.nc ... WOA_12.nc found in {WOA_DIR}')

# Construct explicit month=1...12 axis, preserving missing monthly files as NaN.
woa_template = next(iter(woa_by_month.values()))
woa_clim = xr.concat([woa_by_month.get(m, xr.full_like(woa_template, np.nan))
                      .expand_dims(month=[m]) for m in range(1, 13)], dim='month',
                     join='outer', compat='override', coords='minimal')
# ------------------------------------------------------------
# Mediterranean exclusion (shared by ALL following analyses)
# ------------------------------------------------------------
# Regional water-basin mask designed for the NE Atlantic + western
# Mediterranean. The northern edge tracks the Iberian/French/Italian
# coast approximately; this is *not* a precise coastline dataset.
# Gibraltar longitude ~ -5.6° ensures the Atlantic side is retained.
#
# Longitude / latitude vertices: an approximate Mediterranean exclusion
# region. Coastal land points are already missing in ocean products.
MEDITERRANEAN_EXCLUSION = Polygon([
    (-5.6, 30.0),
    (37.0, 30.0),
    (37.0, 47.0),
    (10.0, 46.0),
    (8.0, 45.0),
    (6.0, 44.0),
    (4.0, 43.65),
    (2.0, 43.35),
    (0.0, 43.05),
    (-5.6, 43.05),
])

def mediterranean_ocean_mask(dataset):
    """2-D keep mask on a regular lat/lon grid; True = not Mediterranean."""
    longitude = np.asarray(dataset.longitude.values, dtype=float)
    latitude = np.asarray(dataset.latitude.values, dtype=float)
    lon2, lat2 = np.meshgrid(longitude, latitude)
    # Longitude convention normalized to [-180, 180) for comparison.
    lon2 = ((lon2 + 180) % 360) - 180
    inside = intersects_xy(MEDITERRANEAN_EXCLUSION, lon2, lat2)
    keep = ~inside
    return xr.DataArray(
        keep, dims=('latitude', 'longitude'),
        coords={'latitude': dataset.latitude, 'longitude': dataset.longitude}
    )

EXCLUDE_MEDITERRANEAN = bool(mediterranean_checkbox.value)

# Keep this function available for map grid markers and later diagnostics.
def apply_mediterranean_mask(dataset, label):
    keep = mediterranean_ocean_mask(dataset)
    masked_n = int((~keep).sum().item())
    print(f'{label}: Mediterranean exclusion: {masked_n} horizontal grid points')
    # Mask every variable: all subsequent analyses will consistently ignore
    # Mediterranean values in maps, means, SDs, profiles and Hovmöllers.
    return dataset.where(keep)

if EXCLUDE_MEDITERRANEAN:
    ds_monthly = apply_mediterranean_mask(ds_monthly, 'CMEMS')
    woa_clim = apply_mediterranean_mask(woa_clim, 'WOA23 monthly')
    if woa_annual is not None:
        woa_annual = apply_mediterranean_mask(woa_annual, 'WOA23 annual')
else:
    print('Mediterranean exclusion: OFF. All loaded ocean basins retained.')

# Download bounding box and visualization geometry are independent.
# No automatic geographic cropping is applied here.
print(f'CMEMS: {len(selected)} of {len(validated)} complete files: '
      f'{pd.Timestamp(ds_monthly.time.values[0]):%Y-%m} -> '
      f'{pd.Timestamp(ds_monthly.time.values[-1]):%Y-%m}')
print('WOA monthly loaded:', sorted(woa_by_month), '| annual:', woa_annual is not None)
print('CMEMS dimensions:', dict(ds_monthly.sizes))
print('WOA dimensions:', dict(woa_clim.sizes))
display(ds_monthly)
display(woa_clim)
# The actual loaded CMEMS area is the authoritative geographic envelope.
# It is NOT inferred from croco_grd.nc; the two areas may differ.
DOWNLOAD_BOUNDS = (
    float(ds_monthly.longitude.min()), float(ds_monthly.longitude.max()),
    float(ds_monthly.latitude.min()), float(ds_monthly.latitude.max())
)
print('Loaded CMEMS envelope [W, E, S, N]:', DOWNLOAD_BOUNDS)
# The model geometry is always independent of the extent of CMEMS downloads.
# Re-apply here to check that all selected bands lie inside the loaded coverage.
apply_geometry()
print('Visualization/model envelope [W, E, S, N]:', MODEL_BOUNDS)
print('Downloaded CMEMS envelope [W, E, S, N]:', DOWNLOAD_BOUNDS)
print('The downloaded coverage should extend outside the model boundaries.')


## 3 — Map: model perimeter, sampling bands, and actual downloaded grids

The **solid black line** denotes the original model/study-domain perimeter (manual coordinates or NetCDF). The **dashed grey line** denotes the actual downloaded CMEMS extent. The sampling bands are centred on the *model boundaries*, **not** on the downloaded-data limits. If the CMEMS and WOA23 dots do not cover a band, verify the files chosen in Section 1 before running scientific analyses.


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
pc=ccrs.PlateCarree()

def draw_model_overlay(ax, draw_bands=False):
    """Plot real model perimeter, distinct from the downloaded-data envelope."""
    if draw_bands:
        for name, poly in BOUNDARY_POLYGONS.items():
            coords = np.asarray(poly.exterior.coords)
            ax.fill(coords[:,0], coords[:,1], facecolor=BOUNDARY_COLORS[name],
                    alpha=0.17, edgecolor='none', transform=pc, zorder=3)
            ax.plot(coords[:,0], coords[:,1], color=BOUNDARY_COLORS[name],
                    lw=1.0, alpha=0.8, transform=pc, zorder=4)
    if DOWNLOAD_BOUNDS is not None:
        w, e, s, n = DOWNLOAD_BOUNDS
        perimeter = np.array([(w,s),(e,s),(e,n),(w,n),(w,s)])
        ax.plot(perimeter[:,0], perimeter[:,1], color='#555555', ls='--',
                lw=1.5, transform=pc, zorder=7)
    coords = np.asarray(MODEL_OUTLINE.exterior.coords)
    ax.plot(coords[:,0], coords[:,1], ls='-', color='black', lw=2.1,
            transform=pc, zorder=9)

def setup_domain_map(ax, extra=1.0):
    lon0,lon1,lat0,lat1 = MODEL_BOUNDS
    ax.set_extent([lon0-extra,lon1+extra,lat0-extra,lat1+extra],crs=pc)
    ax.set_facecolor('#e6f3fa')
    ax.add_feature(cfeature.LAND.with_scale('50m'),facecolor='#e0e0e0',zorder=5)
    ax.coastlines(resolution='50m',linewidth=.8,color='#555',zorder=6)
    grid=ax.gridlines(draw_labels=True,alpha=.25,linestyle=':',linewidth=.6)
    grid.top_labels=grid.right_labels=False

if MODEL_OUTLINE is None:
    raise RuntimeError("Apply visualization geometry in Section 1 before plotting.")
fig=plt.figure(figsize=(14,9))
ax=fig.add_subplot(111,projection=pc)
# Show all boundary strips with slight margin, not just the model rectangle.
bb = np.array([p.bounds for p in BOUNDARY_POLYGONS.values()])
view_boxes = list(bb) + ([ (DOWNLOAD_BOUNDS[0], DOWNLOAD_BOUNDS[2], DOWNLOAD_BOUNDS[1], DOWNLOAD_BOUNDS[3]) ] if DOWNLOAD_BOUNDS is not None else [])
ax.set_extent([min(b[0] for b in view_boxes)-0.3,max(b[2] for b in view_boxes)+0.3,
               min(b[1] for b in view_boxes)-0.3,max(b[3] for b in view_boxes)+0.3],crs=pc)
ax.set_facecolor('#e6f3fa')
clon,clat=np.meshgrid(ds_monthly.longitude.values,ds_monthly.latitude.values)
cm_keep = mediterranean_ocean_mask(ds_monthly).values if EXCLUDE_MEDITERRANEAN else np.ones(clon.shape, dtype=bool)
ax.scatter(clon[cm_keep],clat[cm_keep],marker='.',s=4,color='#4387ad',alpha=.4,
           transform=pc,zorder=2,rasterized=True)
wlon,wlat=np.meshgrid(woa_clim.longitude.values,woa_clim.latitude.values)
wo_keep = mediterranean_ocean_mask(woa_clim).values if EXCLUDE_MEDITERRANEAN else np.ones(wlon.shape, dtype=bool)
ax.scatter(wlon[wo_keep],wlat[wo_keep],marker='s',s=15,facecolors='none',edgecolors='#c77829',lw=.6,
           transform=pc,zorder=2,rasterized=True)
ax.add_feature(cfeature.LAND.with_scale('50m'),facecolor='#dddddd',zorder=5)
ax.coastlines('50m',lw=.8,color='#555',zorder=6)
draw_model_overlay(ax,draw_bands=True)
gl=ax.gridlines(draw_labels=True,alpha=.3,linestyle=':')
gl.top_labels=gl.right_labels=False
legend=[Line2D([0],[0],ls='-',lw=2,color='black',label='Model/visualization boundary'),
        Line2D([0],[0],ls='--',lw=1.5,color='#555555',label='CMEMS downloaded extent')]
legend += [Patch(facecolor=BOUNDARY_COLORS[k],alpha=.5,label=f'{k} sampling band') for k in REGIONS]
legend += [Line2D([0],[0],ls='none',marker='.',color='#4387ad',label='CMEMS grid'),
           Line2D([0],[0],ls='none',marker='s',mfc='none',mec='#c77829',label='WOA23 grid')]
ax.legend(handles=legend,bbox_to_anchor=(1.02,1),loc='upper left',fontsize=10)
mask_label = ' | Mediterranean excluded' if EXCLUDE_MEDITERRANEAN else ''
ax.set_title(f'Visualization geometry ({GEOMETRY_MODE}) | boundary sampling bands{mask_label}',fontsize=15)
plt.show()


## 4 — Interactive CMEMS/WOA23 monthly climatology maps and anomalies

Choose month and depth range. Maps display the **visualization geometry**, which may differ from the download box.

In [ ]:
# ============================================================
# SECTION 4 — MONTHLY CLIMATOLOGIES: CMEMS VS WOA23
# ============================================================

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import ipywidgets as widgets

from io import BytesIO
from IPython.display import display
from matplotlib.patches import Rectangle


# ============================================================
# 1. CONFIGURATION
# ============================================================

pc = ccrs.PlateCarree()

VARIABLES = {
    "no3": "Nitrate",
    "po4": "Phosphate",
    "si": "Silicate",
    "o2": "Dissolved oxygen"
}

MONTH_NAMES = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

UNITS_LABEL = "mmol m$^{-3}$"

MAP_EXTENT = [MODEL_BOUNDS[0]-2, MODEL_BOUNDS[1]+2,
              MODEL_BOUNDS[2]-2, MODEL_BOUNDS[3]+2]


# ============================================================
# 2. GEOGRAPHIC MAP
# ============================================================

def setup_map(ax):

    ax.set_extent(MAP_EXTENT, crs=pc)

    ax.set_facecolor("#e6f3fa")

    gl = ax.gridlines(
        crs=pc,
        draw_labels=True,
        linewidth=0.5,
        color="gray",
        alpha=0.4,
        linestyle=":"
    )

    gl.top_labels = False
    gl.right_labels = False

    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}


def add_map_overlay(ax):

    # Continents
    ax.add_feature(
        cfeature.LAND.with_scale("50m"),
        facecolor="#dddddd",
        edgecolor="none",
        zorder=5
    )

    # Coastlines
    ax.coastlines(
        resolution="50m",
        linewidth=0.8,
        color="black",
        zorder=6
    )

    # Actual visualization outline; not the download bounding box
    draw_model_overlay(ax, draw_bands=False)


# ============================================================
# 3. VERTICAL WEIGHTING
# ============================================================

def depth_weights(depth, zmin, zmax):
    """
    Calculate vertical weights using layer thickness.

    Layer boundaries are estimated from the midpoints
    between consecutive model depth levels.

    Only the part of each layer overlapping the selected
    depth interval contributes to the mean.
    """

    z = np.asarray(depth.values, dtype=float)

    if len(z) < 2:
        raise ValueError(
            "At least two depth levels are required."
        )

    # Midpoints between depth levels
    edges = np.empty(len(z) + 1)

    edges[1:-1] = 0.5 * (z[:-1] + z[1:])

    # Surface boundary
    edges[0] = 0.0

    # Bottom boundary estimated from last spacing
    edges[-1] = z[-1] + 0.5 * (z[-1] - z[-2])

    # Thickness within selected interval
    weights = np.maximum(
        0.0,
        np.minimum(edges[1:], zmax)
        - np.maximum(edges[:-1], zmin)
    )

    return xr.DataArray(
        weights,
        coords={"depth": depth},
        dims=["depth"]
    )


def vertical_mean(da, zmin, zmax):
    """
    Thickness-weighted vertical mean.

    NaN values are excluded.
    No vertical extrapolation is performed.
    """

    weights = depth_weights(
        da.depth,
        zmin,
        zmax
    )

    # Restrict to overlapping depth levels
    valid = weights > 0

    da = da.sel(
        depth=weights.depth.where(valid, drop=True)
    )

    weights = weights.where(
        valid,
        drop=True
    )

    if da.sizes.get("depth", 0) == 0:
        raise ValueError(
            "No depth levels overlap the selected range."
        )

    return da.weighted(weights).mean(
        dim="depth",
        skipna=True
    )


# ============================================================
# 4. MONTHLY CLIMATOLOGIES
# ============================================================

def cmems_climatology(month, zmin, zmax):
    """
    Calculate CMEMS monthly climatology using
    all available years for the selected month.
    """

    selected = ds_monthly.where(
        ds_monthly.time.dt.month == month,
        drop=True
    )

    if selected.sizes.get("time", 0) == 0:
        raise ValueError(
            f"No CMEMS data available for {MONTH_NAMES[month-1]}"
        )

    # Vertical mean first, to reduce data volume
    fields = {}

    for var in VARIABLES:

        da = vertical_mean(
            selected[var],
            zmin,
            zmax
        )

        # Average all available years
        fields[var] = da.mean(
            dim="time",
            skipna=True
        ).compute()

    return fields, selected.sizes["time"]


def woa_climatology(month, zmin, zmax):
    """
    Select WOA23 monthly climatology and calculate
    the vertical mean over the requested range.
    """

    selected = woa_clim.sel(
        month=month,
        drop=True
    )

    fields = {}

    for var in VARIABLES:

        fields[var] = vertical_mean(
            selected[var],
            zmin,
            zmax
        ).compute()

    return fields


# ============================================================
# 5. FIGURE EXPORT
# ============================================================

def figure_to_png(fig):
    """
    Convert a Matplotlib figure to PNG bytes.

    The two figures will be displayed using separate
    ipywidgets.Image widgets.
    """

    buffer = BytesIO()

    try:

        fig.savefig(
            buffer,
            format="png",
            dpi=140,
            bbox_inches="tight",
            facecolor="white"
        )

        return buffer.getvalue()

    finally:

        buffer.close()
        plt.close(fig)


# ============================================================
# 6. FIGURE 1 — CMEMS VS WOA23
# ============================================================

def plot_climatology_maps(
    cm_fields,
    wo_fields,
    month,
    zmin,
    zmax
):

    fig, axes = plt.subplots(
        4,
        2,
        figsize=(15, 19),
        subplot_kw={"projection": pc},
        constrained_layout=True
    )

    for row, (var, label) in enumerate(
        VARIABLES.items()
    ):

        cm = cm_fields[var]
        wo = wo_fields[var]

        # Shared color scale for each variable
        combined = np.concatenate([
            np.asarray(cm.values).ravel(),
            np.asarray(wo.values).ravel()
        ])

        combined = combined[
            np.isfinite(combined)
        ]

        if combined.size:

            vmin, vmax = np.nanpercentile(
                combined,
                [2, 98]
            )

        else:

            vmin, vmax = 0.0, 1.0

        if vmax <= vmin:
            vmax = vmin + 1.0

        for col, (da, product) in enumerate([
            (cm, "CMEMS"),
            (wo, "WOA23")
        ]):

            ax = axes[row, col]

            setup_map(ax)

            da.plot.pcolormesh(
                ax=ax,
                x="longitude",
                y="latitude",
                transform=pc,
                cmap="turbo",
                vmin=vmin,
                vmax=vmax,
                cbar_kwargs={
                    "label": UNITS_LABEL,
                    "shrink": 0.75
                },
                zorder=2
            )

            add_map_overlay(ax)

            ax.set_title(
                f"{label} | {product}",
                fontsize=11
            )

    fig.suptitle(
        f"Monthly climatology | {MONTH_NAMES[month-1]} | "
        f"{zmin:.0f}–{zmax:.0f} m",
        fontsize=16
    )

    return figure_to_png(fig)


# ============================================================
# 7. FIGURE 2 — CLIMATOLOGICAL ANOMALIES
# ============================================================

def plot_anomaly_maps(
    cm_fields,
    wo_fields,
    month,
    zmin,
    zmax
):

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15, 10),
        subplot_kw={"projection": pc},
        constrained_layout=True
    )

    for ax, (var, label) in zip(
        axes.flat,
        VARIABLES.items()
    ):

        cm = cm_fields[var]
        wo = wo_fields[var]

        # Interpolate WOA23 horizontally
        # onto the CMEMS grid

        wo_interp = wo.interp(
            latitude=np.asarray(cm.latitude.values),
            longitude=np.asarray(cm.longitude.values),
            method="linear",
            kwargs={"fill_value": np.nan}
        )

        # CMEMS - WOA23
        anomaly = cm - wo_interp

        values = np.asarray(
            anomaly.values
        )

        finite = np.abs(
            values[np.isfinite(values)]
        )

        if finite.size:

            scale = float(
                np.nanpercentile(finite, 98)
            )

        else:

            scale = 1.0

        if not np.isfinite(scale) or scale <= 0:
            scale = 1.0

        setup_map(ax)

        anomaly.plot.pcolormesh(
            ax=ax,
            x="longitude",
            y="latitude",
            transform=pc,
            cmap="RdBu_r",
            vmin=-scale,
            vmax=scale,
            cbar_kwargs={
                "label": f"CMEMS − WOA23 ({UNITS_LABEL})",
                "shrink": 0.75
            },
            zorder=2
        )

        add_map_overlay(ax)

        ax.set_title(
            f"{label} | CMEMS − WOA23",
            fontsize=11
        )

    fig.suptitle(
        f"Climatological anomalies | {MONTH_NAMES[month-1]} | "
        f"{zmin:.0f}–{zmax:.0f} m",
        fontsize=16
    )

    return figure_to_png(fig)


# ============================================================
# 8. INTERACTIVE CONTROLS
# ============================================================

month_selector = widgets.Dropdown(
    options=[
        (name, i + 1)
        for i, name in enumerate(MONTH_NAMES)
    ],
    value=1,
    description="Month:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="210px")
)

depth_min_selector = widgets.BoundedFloatText(
    value=0.0,
    min=0.0,
    max=5900.0,
    step=10.0,
    description="Min depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

depth_max_selector = widgets.BoundedFloatText(
    value=50.0,
    min=0.0,
    max=5900.0,
    step=10.0,
    description="Max depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

plot_button = widgets.Button(
    description="Generate maps",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="170px")
)

# Separate image widgets:
# Both figures remain visible independently.
image_climatology = widgets.Image(format="png")
image_anomaly = widgets.Image(format="png")

status_label = widgets.HTML(
    value="<i>Select a month and depth range.</i>"
)


# ============================================================
# 9. BUTTON CALLBACK
# ============================================================

def on_plot_clicked(button):

    plot_button.disabled = True

    try:

        month = int(month_selector.value)

        zmin = float(depth_min_selector.value)
        zmax = float(depth_max_selector.value)

        if zmin >= zmax:
            raise ValueError(
                "Maximum depth must be greater than minimum depth."
            )

        status_label.value = (
            "<b>Calculating monthly climatologies...</b>"
        )

        # Calculate CMEMS climatology
        cm_fields, n_years = cmems_climatology(
            month,
            zmin,
            zmax
        )

        # Calculate WOA23 climatology
        wo_fields = woa_climatology(
            month,
            zmin,
            zmax
        )

        # First figure
        image_climatology.value = plot_climatology_maps(
            cm_fields,
            wo_fields,
            month,
            zmin,
            zmax
        )

        # Second figure
        image_anomaly.value = plot_anomaly_maps(
            cm_fields,
            wo_fields,
            month,
            zmin,
            zmax
        )

        status_label.value = (
            f"<b>Completed.</b> "
            f"CMEMS climatology calculated using "
            f"{n_years} monthly records for "
            f"{MONTH_NAMES[month-1]}."
        )

    except Exception as exc:

        status_label.value = (
            f"<b style='color:red'>Error:</b> {exc}"
        )

        raise

    finally:

        plot_button.disabled = False


plot_button.on_click(on_plot_clicked)


# ============================================================
# 10. DISPLAY INTERACTIVE PANEL
# ============================================================

controls = widgets.HBox([
    guided_control(month_selector, 'Select a calendar month (January–December), averaged across available CMEMS years.'),
    guided_control(depth_min_selector, 'Shallowest depth (m) of the layer to average.'),
    guided_control(depth_max_selector, 'Deepest depth (m). Must exceed Min depth; only available levels contribute.'),
    guided_control(plot_button, 'Generate maps and the CMEMS minus WOA23 anomaly.')
], layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

display(
    widgets.HTML(
        "<h3>CMEMS / WOA23 — Monthly climatologies</h3>"
    ),
    widgets.HTML('<div style="padding:9px 12px;margin:4px 0 12px;background:#f2f6fb;border-left:3px solid #557ca3;color:#253a4b;"><b>What you will see:</b> Monthly CMEMS and WOA23 maps for the selected depth layer, followed by CMEMS − WOA23. Depths are in metres (positive downward).</div>'),
    controls,
    status_label,
    widgets.HTML(
        "<h4>Figure 1 — CMEMS and WOA23 climatologies</h4>"
    ),
    image_climatology,
    widgets.HTML(
        "<h4>Figure 2 — CMEMS minus WOA23 anomalies</h4>"
    ),
    image_anomaly
)

## 5 — Geometry-aware boundary sampling and statistics

Each active boundary is represented by a polygon band along the **actual boundary line**. The same geographic masks are used for profiles, seasonal climatologies, time series and Hovmöllers. The selected band width is specified in Section 1. Longitude/latitude degrees give a coordinate-based buffer, not constant kilometres; choose km for physically consistent distances. Download coverage and the underlying CROCO grid are independent.

In [ ]:
# The same geometry-aware region_view is used in all analysis sections.
# Cached 2-D masks are very small and keyed by the exact source grid.
_BOUNDARY_MASK_CACHE = {}

def _identify_boundary(region):
    if isinstance(region,str):
        return region,BOUNDARY_POLYGONS[region]
    for name,poly in REGIONS.items():
        if region is poly:
            return name,poly
    if hasattr(region,'geom_type'):
        return 'custom',region
    raise TypeError('Expected a registered boundary name or geometry polygon.')

def boundary_mask(da, region):
    name,poly=_identify_boundary(region)
    lat=np.asarray(da.latitude.values,dtype=float)
    lon=np.asarray(da.longitude.values,dtype=float)
    key=(name,hash(lat.tobytes()),hash(lon.tobytes()),poly.wkb)
    if key not in _BOUNDARY_MASK_CACHE:
        xx,yy=np.meshgrid(lon,lat)
        mask=intersects_xy(poly,xx,yy)
        if not mask.any():
            raise ValueError(f'Boundary {name!r} contains no grid cells. Check geometry and data coverage.')
        _BOUNDARY_MASK_CACHE[key]=xr.DataArray(mask,dims=('latitude','longitude'),
            coords={'latitude':da.latitude,'longitude':da.longitude})
    return _BOUNDARY_MASK_CACHE[key]

def region_view(obj, region):
    _,poly=_identify_boundary(region)
    x0,y0,x1,y1=poly.bounds
    # First crop to boundary bbox to avoid expanding masks across full model grids.
    sub=obj.sel(longitude=slice(x0-0.01,x1+0.01),latitude=slice(y0-0.01,y1+0.01))
    if not sub.sizes.get('longitude',0) or not sub.sizes.get('latitude',0):
        raise ValueError('No downloaded data cover this boundary band.')
    return sub.where(boundary_mask(sub,region))

def spatial_stats(da):
    # weighted result excludes missing values and land NaNs
    w=np.cos(np.deg2rad(da.latitude)).clip(min=0)
    weighted=da.weighted(w)
    return weighted.mean(('latitude','longitude')),weighted.std(('latitude','longitude'))

def depth_edges(depth):
    z = np.asarray(depth.values, dtype=float)
    if len(z) < 2:
        raise ValueError('At least 2 depth coordinates are required')
    edges = np.empty(len(z)+1)
    edges[1:-1] = (z[1:]+z[:-1])/2
    edges[0] = max(0., z[0]-(z[1]-z[0])/2)
    edges[-1] = z[-1]+(z[-1]-z[-2])/2
    return edges

def layer_stats(da, z_min, z_max):
    if z_max <= z_min:
        raise ValueError('Depth range must satisfy lower bound < upper bound')
    edges = depth_edges(da.depth)
    thickness = np.maximum(0., np.minimum(edges[1:], z_max)-np.maximum(edges[:-1], z_min))
    if not np.any(thickness > 0):
        # Explicit NaNs rather than silently using the closest depth.
        empty = da.isel(depth=0, drop=True).mean(('latitude', 'longitude')) * np.nan
        return empty, empty.copy()
    t = xr.DataArray(thickness, dims=('depth',), coords={'depth': da.depth})
    weights = t * np.cos(np.deg2rad(da.latitude))
    weighted = da.where(t > 0).weighted(weights)
    dims = ('depth', 'latitude', 'longitude')
    return weighted.mean(dims), weighted.std(dims)

def boundary_layer(da, bounds, z_range):
    return layer_stats(region_view(da, bounds), *z_range)

# Convenience function: calculate CMEMS monthly climatological statistics using every
# available year. First reduce the large fields to low-memory boundary time series.
def cmems_clim_layer(variable, bounds, z_range):
    mu, sd = boundary_layer(ds_monthly[variable], bounds, z_range)
    # Climatological mean: average year-to-year boundary means for each calendar month.
    # SD is the temporal mean of the within-month spatial/vertical SD, not interannual SD.
    clim_mu = mu.groupby('time.month').mean('time', skipna=True).reindex(month=np.arange(1,13))
    clim_sd = sd.groupby('time.month').mean('time', skipna=True).reindex(month=np.arange(1,13))
    return clim_mu, clim_sd

def woa_clim_layer(variable, bounds, z_range):
    return boundary_layer(woa_clim[variable], bounds, z_range)

print('Statistics ready. Active boundary bands:', list(REGIONS))

## 6 — Climatological vertical profiles by active boundary

Interactive month and maximum depth. CMEMS and WOA23 retain native vertical levels; band selection uses the geometry from Section 1.

In [ ]:
# ============================================================
# SECTION 6 — INTERACTIVE CLIMATOLOGICAL VERTICAL PROFILES
# CMEMS vs WOA23 | 4 variables x 4 boundaries
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from dask import compute
from io import BytesIO
from IPython.display import display, HTML
from IPython.display import Image as IPImage


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROFILE_MAX_DEPTH = 2000.0

MONTH_NAMES = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

VARIABLES = {
    "no3": "Nitrate",
    "po4": "Phosphate",
    "si": "Silicate",
    "o2": "Dissolved oxygen"
}

# Product colors
CMEMS_COLOR = "tab:blue"
WOA_COLOR = "tab:orange"

# Spatial variability shading
SHADE_ALPHA = 0.18


# ============================================================
# 2. CALCULATE MONTHLY CLIMATOLOGICAL PROFILES
# ============================================================

def climatological_profiles(month, max_depth):

    """
    Calculate monthly climatological vertical profiles
    for CMEMS and WOA23 in the four boundary regions.

    CMEMS:
        1. Select all available years for the chosen month.
        2. Calculate the temporal climatological mean.
        3. Calculate spatial mean and spatial standard
           deviation inside each boundary region.

    WOA23:
        1. Select the monthly climatology.
        2. Calculate spatial mean and spatial standard
           deviation inside each boundary region.

    No vertical interpolation is performed.

    Both products retain their original depth levels.
    """

    # --------------------------------------------------------
    # Select CMEMS months
    # --------------------------------------------------------

    cm_selected = ds_monthly.where(
        ds_monthly.time.dt.month == month,
        drop=True
    )

    n_years = cm_selected.sizes.get("time", 0)

    if n_years == 0:
        raise ValueError(
            f"No CMEMS data available for {MONTH_NAMES[month-1]}"
        )

    # --------------------------------------------------------
    # Select WOA23 climatology
    # --------------------------------------------------------

    wo_selected = woa_clim.sel(
        month=month,
        drop=True
    )

    # --------------------------------------------------------
    # Calculate profiles
    # --------------------------------------------------------

    results = {}

    for var in VARIABLES:

        results[var] = {}

        for boundary, bounds in REGIONS.items():

            # -----------------------------------------------
            # CMEMS
            # -----------------------------------------------

            cm_region = region_view(
                cm_selected[var],
                bounds
            )

            cm_region = cm_region.sel(
                depth=slice(None, max_depth)
            )

            # Monthly climatology across available years
            cm_clim = cm_region.mean(
                dim="time",
                skipna=True
            )

            # Spatial mean and standard deviation
            cm_mean, cm_std = spatial_stats(
                cm_clim
            )

            # -----------------------------------------------
            # WOA23
            # -----------------------------------------------

            wo_region = region_view(
                wo_selected[var],
                bounds
            )

            wo_region = wo_region.sel(
                depth=slice(None, max_depth)
            )

            wo_mean, wo_std = spatial_stats(
                wo_region
            )

            # -----------------------------------------------
            # Compute results
            # -----------------------------------------------

            cm_mean, cm_std, wo_mean, wo_std = compute(
                cm_mean,
                cm_std,
                wo_mean,
                wo_std
            )

            results[var][boundary] = {

                "cm_mean": cm_mean,
                "cm_std": cm_std,

                "wo_mean": wo_mean,
                "wo_std": wo_std

            }

    return results, n_years


# ============================================================
# 3. PLOTTING FUNCTION
# ============================================================

def plot_climatological_profiles(month, max_depth):

    """
    Plot a 4x4 figure:

    Rows:
        NO3, PO4, Si, O2

    Columns:
        North, South, West, East

    Each subplot:
        CMEMS climatological mean +/- spatial SD
        WOA23 climatological mean +/- spatial SD
    """

    # Calculate profiles
    results, n_years = climatological_profiles(
        month,
        max_depth
    )

    boundary_names = list(REGIONS.keys())

    variable_names = list(VARIABLES.keys())

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    nbound = len(boundary_names)
    fig, axes = plt.subplots(
        4, nbound, figsize=(max(15, 5.3*nbound), 20),
        sharey=True, constrained_layout=True, squeeze=False
    )

    # --------------------------------------------------------
    # Loop over variables and boundaries
    # --------------------------------------------------------

    for row, var in enumerate(variable_names):

        for col, boundary in enumerate(boundary_names):

            ax = axes[row, col]

            data = results[var][boundary]

            # -----------------------------------------------
            # CMEMS
            # -----------------------------------------------

            cm_mean = data["cm_mean"]
            cm_std = data["cm_std"]

            z_cm = cm_mean.depth.values

            ax.plot(
                cm_mean.values,
                z_cm,
                color=CMEMS_COLOR,
                linewidth=2,
                linestyle="-",
                label="CMEMS"
            )

            ax.fill_betweenx(
                z_cm,
                (cm_mean - cm_std).values,
                (cm_mean + cm_std).values,
                color=CMEMS_COLOR,
                alpha=SHADE_ALPHA,
                linewidth=0
            )

            # -----------------------------------------------
            # WOA23
            # -----------------------------------------------

            wo_mean = data["wo_mean"]
            wo_std = data["wo_std"]

            z_wo = wo_mean.depth.values

            ax.plot(
                wo_mean.values,
                z_wo,
                color=WOA_COLOR,
                linewidth=2,
                linestyle="--",
                label="WOA23"
            )

            ax.fill_betweenx(
                z_wo,
                (wo_mean - wo_std).values,
                (wo_mean + wo_std).values,
                color=WOA_COLOR,
                alpha=SHADE_ALPHA,
                linewidth=0
            )

            #            # -----------------------------------------------
            # Axis configuration
            # -----------------------------------------------

            ax.set_ylim(max_depth, 0)

            ax.grid(
                True,
                alpha=0.25,
                linestyle=":"
            )

            # Larger tick labels
            ax.tick_params(
                axis="both",
                labelsize=14,
                width=1.2,
                length=5
            )

            # Column titles
            if row == 0:
                ax.set_title(
                    f"{boundary} boundary",
                    fontsize=18,
                    fontweight="bold",
                    pad=12
                )

            # Y-axis labels
            if col == 0:
                ax.set_ylabel(
                    f"{VARIABLES[var]}\nDepth (m)",
                    fontsize=17,
                    fontweight="bold"
                )

            # X-axis labels
            if row == 3:
                ax.set_xlabel(
                    "Concentration (mmol m$^{-3}$)",
                    fontsize=15,
                    labelpad=10
                )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    axes[0, 0].legend(
        loc="best",
        fontsize=15,
        frameon=True
    )

    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        f"Climatological vertical profiles | "
        f"{MONTH_NAMES[month-1]}\n"
        f"CMEMS vs WOA23 | "
        f"Mean ± spatial SD",
        fontsize=21,
        fontweight="bold"
    )

    # --------------------------------------------------------
    # Convert figure to PNG
    # --------------------------------------------------------

    buffer = BytesIO()

    try:
        fig.savefig(
            buffer,
            format="png",
            dpi=160,
            bbox_inches="tight",
            facecolor="white"
        )

        image_data = buffer.getvalue()

    finally:
        buffer.close()
        plt.close(fig)

    return image_data


# ============================================================
# 4. INTERACTIVE CONTROLS
# ============================================================

profile_month_selector = widgets.Dropdown(
    options=[
        (name, i + 1)
        for i, name in enumerate(MONTH_NAMES)
    ],
    value=1,
    description="Month:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px")
)

profile_depth_selector = widgets.BoundedFloatText(
    value=2000.0,
    min=10.0,
    max=5900.0,
    step=100.0,
    description="Max depth (m):",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="240px")
)

profile_button = widgets.Button(
    description="Generate profiles",
    button_style="primary",
    icon="line-chart",
    layout=widgets.Layout(width="180px")
)

profile_status = widgets.HTML(
    value="<i>Select a month and maximum depth.</i>"
)

profile_image = widgets.Image(
    format="png"
)


# ============================================================
# 5. BUTTON CALLBACK
# ============================================================

def on_profile_clicked(button):

    profile_button.disabled = True

    try:

        month = int(
            profile_month_selector.value
        )

        max_depth = float(
            profile_depth_selector.value
        )

        profile_status.value = (
            "<b>Calculating climatological profiles...</b>"
        )

        image_data = plot_climatological_profiles(
            month,
            max_depth
        )

        profile_image.value = image_data

        profile_status.value = (
            f"<b>Completed:</b> "
            f"{MONTH_NAMES[month-1]}, "
            f"0–{max_depth:.0f} m."
        )

    except Exception as exc:

        profile_status.value = (
            f"<b style='color:red'>Error:</b> {exc}"
        )

        raise

    finally:

        profile_button.disabled = False


profile_button.on_click(
    on_profile_clicked
)


# ============================================================
# 6. DISPLAY INTERACTIVE PANEL
# ============================================================

profile_controls = widgets.HBox([
    guided_control(profile_month_selector, 'Calendar month used to calculate the climatological vertical profiles.'),
    guided_control(profile_depth_selector, 'Deepest depth to display (m); surface is at 0 m.'),
    guided_control(profile_button, 'Plot CMEMS and WOA23 mean profiles and their spatial variability.')
], layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

display(
    widgets.HTML(
        "<h3>CMEMS / WOA23 — Climatological vertical profiles</h3>"
    ),
    widgets.HTML('<div style="padding:9px 12px;margin:4px 0 12px;background:#f2f6fb;border-left:3px solid #557ca3;color:#253a4b;"><b>What you will see:</b> Vertical profiles for each ocean boundary. Curves show spatial means; shaded areas show spatial standard deviations.</div>'),
    profile_controls,
    profile_status,
    profile_image
)

## 7 — Seasonal depth–month Hovmöllers and anomalies

Computed within the active sampling polygons from Section 1.

In [ ]:
# Compute boundary mean at each level, then monthly climatology. Work on reduced arrays.
cm_hov, wo_hov = {}, {}
for name, bounds in REGIONS.items():
    cm_field = region_view(ds_monthly[HOV_VARIABLE], bounds).sel(depth=slice(None, HOV_MAX_DEPTH))
    cm_mean, _ = spatial_stats(cm_field)
    cm_hov[name] = cm_mean.groupby('time.month').mean('time').reindex(month=np.arange(1,13))
    wo_field = region_view(woa_clim[HOV_VARIABLE], bounds).sel(depth=slice(None, HOV_MAX_DEPTH))
    wo_hov[name], _ = spatial_stats(wo_field)
cm_hov = {k: v.compute() for k,v in cm_hov.items()}
wo_hov = {k: v.compute() for k,v in wo_hov.items()}

def hov_season_panel(mode):
    cols=min(2,len(REGIONS)); rows=int(np.ceil(len(REGIONS)/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(7*cols,5*rows),
                             constrained_layout=True, sharex=True, sharey=True,squeeze=False)
    for unused in axes.flat[len(REGIONS):]:unused.set_visible(False)
    for ax, name in zip(axes.flat, REGIONS):
        if mode == 'CMEMS':
            data = cm_hov[name]
        elif mode == 'WOA23':
            data = wo_hov[name]
        else:
            # Compare on native CMEMS z levels, no extrapolation.
            wo_on_cm = wo_hov[name].interp(depth=cm_hov[name].depth,
                                          kwargs={'fill_value': np.nan})
            data = cm_hov[name] - wo_on_cm
        vals = np.asarray(data.values)
        valid = np.abs(vals[np.isfinite(vals)])
        scale = float(np.nanpercentile(valid, 98)) if valid.size else 1.
        if scale <= 0: scale = 1.
        style = ({'cmap': 'RdBu_r', 'vmin': -scale, 'vmax': scale} if mode == 'Anomaly'
                 else {'cmap': 'turbo'})
        data.transpose('depth','month').plot.pcolormesh(
            ax=ax, x='month', y='depth', **style,
            cbar_kwargs={'label': UNITS})
        ax.set(title=name, xlabel='Calendar month', ylabel='Depth (m)',
               xticks=np.arange(1, 13))
        ax.set_ylim(HOV_MAX_DEPTH, 0)
    if mode == 'Anomaly':
        title = f'{VARS[HOV_VARIABLE]} | Monthly climatology anomaly (CMEMS − WOA23) | selected boundaries'
    else:
        title = f'{VARS[HOV_VARIABLE]} | {mode} monthly climatology | selected boundaries'
    
    fig.suptitle(title, fontsize=17, fontweight='bold')
    plt.show()

for mode in ('CMEMS', 'WOA23', 'Anomaly'):
    hov_season_panel(mode)

## 8 — Seasonal climatologies of boundary bands

Interactive variable and depth range; statistics use the same active boundary polygon masks.

In [ ]:
# ============================================================
# SECTION 8 — INTERACTIVE SEASONAL CLIMATOLOGIES
# CMEMS vs WOA23 | Mean ± spatial and vertical SD
# ============================================================

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as widgets

from dask import compute
from io import BytesIO
from IPython.display import display


# ============================================================
# 1. CONFIGURATION
# ============================================================

CLIM_VARIABLES = {
    "no3": "Nitrate",
    "po4": "Phosphate",
    "si": "Silicate",
    "o2": "Dissolved oxygen"
}

MONTH_NAMES = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

MONTHS = np.arange(1, 13)

CMEMS_COLOR = "tab:blue"
WOA_COLOR = "tab:orange"

SHADE_ALPHA = 0.18


# ============================================================
# 2. VERTICAL THICKNESS WEIGHTS
# ============================================================

def climatology_depth_weights(depth, zmin, zmax):
    """
    Calculate layer-thickness weights.

    Only the part of each layer overlapping the selected
    depth interval is included.

    No extrapolation beyond available depth levels.
    """

    z = np.asarray(depth.values, dtype=float)

    edges = np.empty(len(z) + 1)

    edges[1:-1] = 0.5 * (z[:-1] + z[1:])

    edges[0] = 0.0

    edges[-1] = (
        z[-1] + 0.5 * (z[-1] - z[-2])
    )

    # Overlap with requested interval
    weights = np.maximum(
        0.0,
        np.minimum(edges[1:], zmax)
        - np.maximum(edges[:-1], zmin)
    )

    return xr.DataArray(
        weights,
        dims=["depth"],
        coords={"depth": depth}
    )


# ============================================================
# 3. SPATIAL AND VERTICAL STATISTICS
# ============================================================

def climatology_layer_stats(da, zmin, zmax):
    """
    Calculate mean and standard deviation across:

        depth
        latitude
        longitude

    Weights:
        - Vertical layer thickness
        - cos(latitude) horizontal weighting

    Missing values are excluded.

    The standard deviation represents the spatial and
    vertical variability of the climatological field.
    """

    # Vertical weights
    wz = climatology_depth_weights(
        da.depth,
        zmin,
        zmax
    )

    # Latitude weights
    wlat = xr.DataArray(
        np.cos(np.deg2rad(da.latitude.values)),
        dims=["latitude"],
        coords={"latitude": da.latitude}
    )

    # Combined weights
    weights = wz * wlat

    # Broadcast weights to depth, latitude and longitude
    weights = weights.broadcast_like(
        da.isel(month=0, drop=True)
        if "month" in da.dims
        else da
    )

    # Avoid zero-weight cells
    weights = weights.where(weights > 0, 0)

    # Weighted mean
    dims = ["depth", "latitude", "longitude"]

    mean = da.weighted(weights).mean(
        dim=dims,
        skipna=True
    )

    # Weighted standard deviation
    std = da.weighted(weights).std(
        dim=dims,
        skipna=True
    )

    return mean, std


# ============================================================
# 4. CMEMS MONTHLY CLIMATOLOGY
# ============================================================

def seasonal_cmems_stats(variable, bounds, zmin, zmax):
    """
    Calculate CMEMS monthly climatology first,
    then spatial and vertical mean and SD.
    """

    # Select boundary
    da = region_view(
        ds_monthly[variable],
        bounds
    )

    # Monthly climatology across available years
    clim = da.groupby(
        "time.month"
    ).mean(
        dim="time",
        skipna=True
    )

    clim = clim.reindex(
        month=MONTHS
    )

    # Spatial and vertical statistics
    mean, std = climatology_layer_stats(
        clim,
        zmin,
        zmax
    )

    return mean, std


# ============================================================
# 5. WOA23 MONTHLY CLIMATOLOGY
# ============================================================

def seasonal_woa_stats(variable, bounds, zmin, zmax):
    """
    Calculate spatial and vertical mean and SD from
    the WOA23 monthly climatology.
    """

    # Select boundary
    da = region_view(
        woa_clim[variable],
        bounds
    )

    da = da.reindex(
        month=MONTHS
    )

    # Spatial and vertical statistics
    mean, std = climatology_layer_stats(
        da,
        zmin,
        zmax
    )

    return mean, std


# ============================================================
# 6. MAIN PLOTTING FUNCTION
# ============================================================

def plot_seasonal_climatologies(variable, zmin, zmax):

    if zmin >= zmax:
        raise ValueError(
            "Maximum depth must be greater than minimum depth."
        )

    # --------------------------------------------------------
    # Calculate statistics
    # --------------------------------------------------------

    results = {}

    for name, bounds in REGIONS.items():

        cm_mu, cm_sd = seasonal_cmems_stats(
            variable,
            bounds,
            zmin,
            zmax
        )

        wo_mu, wo_sd = seasonal_woa_stats(
            variable,
            bounds,
            zmin,
            zmax
        )

        cm_mu, cm_sd, wo_mu, wo_sd = compute(
            cm_mu,
            cm_sd,
            wo_mu,
            wo_sd
        )

        results[name] = {
            "cm_mean": cm_mu,
            "cm_std": cm_sd,
            "wo_mean": wo_mu,
            "wo_std": wo_sd
        }

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    cols = min(2,len(REGIONS)); rows = int(np.ceil(len(REGIONS)/cols))
    fig, axes = plt.subplots(rows,cols,figsize=(8.5*cols,6*rows),
                             sharex=True,constrained_layout=True,squeeze=False)
    for unused in axes.flat[len(REGIONS):]: unused.set_visible(False)

    for ax, (name, data) in zip(
        axes.flat,
        results.items()
    ):

        # ----------------------------------------------------
        # CMEMS
        # ----------------------------------------------------

        cm_mean = np.asarray(
            data["cm_mean"].values
        )

        cm_std = np.asarray(
            data["cm_std"].values
        )

        ax.plot(
            MONTHS,
            cm_mean,
            color=CMEMS_COLOR,
            linewidth=2.5,
            linestyle="-",
            marker="o",
            markersize=5,
            label="CMEMS climatology"
        )

        ax.fill_between(
            MONTHS,
            cm_mean - cm_std,
            cm_mean + cm_std,
            color=CMEMS_COLOR,
            alpha=SHADE_ALPHA,
            linewidth=0
        )

        # ----------------------------------------------------
        # WOA23
        # ----------------------------------------------------

        wo_mean = np.asarray(
            data["wo_mean"].values
        )

        wo_std = np.asarray(
            data["wo_std"].values
        )

        ax.plot(
            MONTHS,
            wo_mean,
            color=WOA_COLOR,
            linewidth=2.5,
            linestyle="--",
            marker="s",
            markersize=5,
            label="WOA23 climatology"
        )

        ax.fill_between(
            MONTHS,
            wo_mean - wo_std,
            wo_mean + wo_std,
            color=WOA_COLOR,
            alpha=SHADE_ALPHA,
            linewidth=0
        )

        # ----------------------------------------------------
        # Axes
        # ----------------------------------------------------

        ax.set_title(
            f"{name} boundary",
            fontsize=18,
            fontweight="bold"
        )

        ax.set_xticks(MONTHS)
        ax.set_xticklabels(MONTH_NAMES)

        ax.set_xlabel(
            "Calendar month",
            fontsize=16
        )

        ax.set_ylabel(
            "Concentration (mmol m$^{-3}$)",
            fontsize=16
        )

        ax.tick_params(
            axis="both",
            labelsize=14
        )

        ax.grid(
            alpha=0.25,
            linestyle=":"
        )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    axes[0, 0].legend(
        loc="best",
        fontsize=14
    )

    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        f"{CLIM_VARIABLES[variable]} | "
        f"Monthly climatology | "
        f"{zmin:.0f}–{zmax:.0f} m\n"
        f"CMEMS vs WOA23 | "
        f"Mean ± spatial and vertical SD",
        fontsize=21,
        fontweight="bold"
    )

    # --------------------------------------------------------
    # Convert figure to PNG
    # --------------------------------------------------------

    buffer = BytesIO()

    try:

        fig.savefig(
            buffer,
            format="png",
            dpi=150,
            bbox_inches="tight",
            facecolor="white"
        )

        return buffer.getvalue()

    finally:

        buffer.close()
        plt.close(fig)


# ============================================================
# 7. INTERACTIVE CONTROLS
# ============================================================

clim_variable_selector = widgets.Dropdown(
    options=[
        (label, var)
        for var, label in CLIM_VARIABLES.items()
    ],
    value="no3",
    description="Variable:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="230px")
)

clim_zmin_selector = widgets.BoundedFloatText(
    value=0.0,
    min=0.0,
    max=5900.0,
    step=10.0,
    description="Min depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

clim_zmax_selector = widgets.BoundedFloatText(
    value=50.0,
    min=0.0,
    max=5900.0,
    step=10.0,
    description="Max depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

clim_button = widgets.Button(
    description="Generate climatology",
    button_style="primary",
    icon="line-chart",
    layout=widgets.Layout(width="210px")
)

clim_status = widgets.HTML(
    value="<i>Select variable and depth range.</i>"
)

clim_image = widgets.Image(
    format="png"
)


# ============================================================
# 8. BUTTON CALLBACK
# ============================================================

def on_clim_clicked(button):

    clim_button.disabled = True

    try:

        variable = clim_variable_selector.value

        zmin = float(
            clim_zmin_selector.value
        )

        zmax = float(
            clim_zmax_selector.value
        )

        clim_status.value = (
            "<b>Calculating climatologies...</b>"
        )

        image_data = plot_seasonal_climatologies(
            variable,
            zmin,
            zmax
        )

        clim_image.value = image_data

        clim_status.value = (
            f"<b>Completed:</b> "
            f"{CLIM_VARIABLES[variable]}, "
            f"{zmin:.0f}–{zmax:.0f} m."
        )

    except Exception as exc:

        clim_status.value = (
            f"<b style='color:red'>Error:</b> {exc}"
        )

        raise

    finally:

        clim_button.disabled = False


clim_button.on_click(
    on_clim_clicked
)


# ============================================================
# 9. DISPLAY PANEL
# ============================================================

controls = widgets.HBox([
    guided_control(clim_variable_selector, 'Choose nitrate, phosphate, silicate or dissolved oxygen.'),
    guided_control(clim_zmin_selector, 'Upper boundary of the depth layer (m; 0 = sea surface).'),
    guided_control(clim_zmax_selector, 'Lower boundary of the depth layer (m). Must be larger than Min depth.'),
    guided_control(clim_button, 'Plot January–December climatological cycles for all active boundaries.')
], layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

display(
    widgets.HTML(
        "<h3>CMEMS / WOA23 — Seasonal climatology comparison</h3>"
    ),
    widgets.HTML('<div style="padding:9px 12px;margin:4px 0 12px;background:#f2f6fb;border-left:3px solid #557ca3;color:#253a4b;"><b>How to use:</b> Choose a variable and a depth interval (for example, 0–50 m), then click Generate climatology. The lines are monthly climatologies, not individual years; shading indicates spatial and vertical variability.</div>'),
    controls,
    clim_status,
    clim_image
)

## 9 — WOA23 pre-1993 + CMEMS since 1993: boundary time series

Select variable, depth interval and time span. Monthly WOA23 where available; annual climatology at depth. This is a repeated climatology, not historical interannual variability.

In [ ]:
# ============================================================
# SECTION 9 — INTERACTIVE WOA23 + CMEMS TIME SERIES
# WOA23: before 1993 | CMEMS: 1993 onward
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from dask import compute
from io import BytesIO
from IPython.display import display
from matplotlib.lines import Line2D


# ============================================================
# 1. CONFIGURATION
# ============================================================

SERIES_VARIABLES = {
    "no3": "Nitrate",
    "po4": "Phosphate",
    "si": "Silicate",
    "o2": "Dissolved oxygen"
}

CMEMS_COLOR = "tab:blue"
WOA_COLOR = "tab:orange"

SHADE_ALPHA = 0.17

FIRST_YEAR = 1960

LAST_YEAR = max(
    2026,
    int(ds_monthly.time.dt.year.max().values)
)

MONTHLY_LIMITS = {
    "no3": 800.0,
    "po4": 800.0,
    "si": 800.0,
    "o2": 1500.0
}

ANNUAL_MAX_DEPTH = 5500.0


# ============================================================
# 2. VALIDATE DEPTH RANGE
# ============================================================

def validate_series_depth(variable, zmin, zmax):

    if variable not in SERIES_VARIABLES:
        raise ValueError(f"Unknown variable: {variable}")

    if zmin < 0 or zmax <= zmin:
        raise ValueError("Depth must satisfy 0 <= min < max.")

    limit = MONTHLY_LIMITS[variable]

    if zmax <= limit:
        return "monthly"

    if zmin >= limit and zmax <= ANNUAL_MAX_DEPTH:
        return "annual"

    raise ValueError(
        f"Invalid depth range: {zmin:.0f}–{zmax:.0f} m. "
        f"Choose 0–{limit:.0f} m (monthly WOA23) or "
        f"{limit:.0f}–5500 m (annual WOA23)."
    )


# ============================================================
# 3. WOA23 STATISTICS
# ============================================================

def series_woa_stats(variable, bounds, zmin, zmax, mode):

    if mode == "monthly":

        da = woa_clim[variable]

    elif mode == "annual":

        if woa_annual is None:
            raise FileNotFoundError(
                "WOA_ANNUAL.nc is not loaded."
            )

        da = woa_annual[variable]

    else:
        raise ValueError("Unknown WOA23 mode.")

    return boundary_layer(
        da,
        bounds,
        (zmin, zmax)
    )


# ============================================================
# 4. CMEMS STATISTICS
# ============================================================

def series_cmems_stats(variable, bounds, zmin, zmax):

    return boundary_layer(
        ds_monthly[variable],
        bounds,
        (zmin, zmax)
    )


# ============================================================
# 5. BUILD COMBINED SERIES
# ============================================================

def build_combined_series(
    variable,
    bounds,
    zmin,
    zmax,
    start_date,
    end_date
):

    mode = validate_series_depth(
        variable,
        zmin,
        zmax
    )

    full_time = pd.date_range(
        start=start_date,
        end=end_date,
        freq="MS"
    )

    # --------------------------------------------------------
    # WOA23 before 1993
    # --------------------------------------------------------

    before = full_time[full_time.year < 1993]

    pre_mean = np.array([])
    pre_std = np.array([])

    if len(before):

        wo_mean, wo_std = series_woa_stats(
            variable,
            bounds,
            zmin,
            zmax,
            mode
        )

        wo_mean, wo_std = compute(
            wo_mean,
            wo_std
        )

        if mode == "monthly":

            wo_mean = wo_mean.reindex(
                month=np.arange(1, 13)
            )

            wo_std = wo_std.reindex(
                month=np.arange(1, 13)
            )

            month_indices = before.month.values - 1

            pre_mean = np.asarray(
                wo_mean.values
            )[month_indices]

            pre_std = np.asarray(
                wo_std.values
            )[month_indices]

        else:

            pre_mean = np.full(
                len(before),
                float(wo_mean.values)
            )

            pre_std = np.full(
                len(before),
                float(wo_std.values)
            )

    # --------------------------------------------------------
    # CMEMS from 1993 onward
    # --------------------------------------------------------

    after = full_time[full_time.year >= 1993]

    post_mean = np.array([])
    post_std = np.array([])

    if len(after):

        cm_mean, cm_std = series_cmems_stats(
            variable,
            bounds,
            zmin,
            zmax
        )

        cm_mean, cm_std = compute(
            cm_mean,
            cm_std
        )

        cm_mean = cm_mean.assign_coords(
            time=pd.to_datetime(cm_mean.time.values)
        )

        cm_std = cm_std.assign_coords(
            time=pd.to_datetime(cm_std.time.values)
        )

        cm_mean = cm_mean.reindex(
            time=after
        )

        cm_std = cm_std.reindex(
            time=after
        )

        post_mean = np.asarray(cm_mean.values)
        post_std = np.asarray(cm_std.values)

    return {
        "mode": mode,
        "before": before,
        "pre_mean": pre_mean,
        "pre_std": pre_std,
        "after": after,
        "post_mean": post_mean,
        "post_std": post_std
    }


# ============================================================
# 6. PLOTTING FUNCTION
# ============================================================

def plot_combined_series(
    variable,
    zmin,
    zmax,
    start_date,
    end_date
):

    mode = validate_series_depth(
        variable,
        zmin,
        zmax
    )

    # --------------------------------------------------------
    # Create figure — manual layout
    # --------------------------------------------------------

    cols=min(2,len(REGIONS)); rows=int(np.ceil(len(REGIONS)/cols))
    fig, axes=plt.subplots(rows,cols,figsize=(9.5*cols,6.5*rows),
                           sharex=True,constrained_layout=False,squeeze=False)
    for unused in axes.flat[len(REGIONS):]:unused.set_visible(False)

    # Reserve a dedicated header area
    fig.subplots_adjust(
        top=0.76,
        bottom=0.09,
        left=0.08,
        right=0.97,
        hspace=0.38,
        wspace=0.23
    )

    # --------------------------------------------------------
    # Four boundaries
    # --------------------------------------------------------

    for ax, (name, bounds) in zip(
        axes.flat,
        REGIONS.items()
    ):

        data = build_combined_series(
            variable,
            bounds,
            zmin,
            zmax,
            start_date,
            end_date
        )

        before = data["before"]
        after = data["after"]

        # ----------------------------------------------------
        # WOA23
        # ----------------------------------------------------

        if len(before):

            y = data["pre_mean"]
            sd = data["pre_std"]

            ax.plot(
                before,
                y,
                color=WOA_COLOR,
                linewidth=2
            )

            ax.fill_between(
                before,
                y - sd,
                y + sd,
                color=WOA_COLOR,
                alpha=SHADE_ALPHA,
                linewidth=0
            )

        # ----------------------------------------------------
        # CMEMS
        # ----------------------------------------------------

        if len(after):

            y = data["post_mean"]
            sd = data["post_std"]

            ax.plot(
                after,
                y,
                color=CMEMS_COLOR,
                linewidth=2
            )

            ax.fill_between(
                after,
                y - sd,
                y + sd,
                color=CMEMS_COLOR,
                alpha=SHADE_ALPHA,
                linewidth=0
            )

        # ----------------------------------------------------
        # Product transition
        # ----------------------------------------------------

        transition = pd.Timestamp("1993-01-01")

        if (
            pd.Timestamp(start_date) < transition
            and pd.Timestamp(end_date) >= transition
        ):

            ax.axvline(
                transition,
                color="black",
                linestyle="--",
                linewidth=1.5
            )

        # ----------------------------------------------------
        # Axes
        # ----------------------------------------------------

        ax.set_title(
            f"{name} boundary",
            fontsize=19,
            fontweight="bold",
            pad=12
        )

        ax.set_ylabel(
            "Concentration (mmol m$^{-3}$)",
            fontsize=16
        )

        ax.set_xlabel(
            "Year",
            fontsize=16
        )

        ax.tick_params(
            axis="both",
            labelsize=14
        )

        ax.grid(
            alpha=0.25,
            linestyle=":"
        )

        ax.set_xlim(
            pd.Timestamp(start_date),
            pd.Timestamp(end_date)
        )

    # ========================================================
    # FIGURE HEADER — TITLE AND LEGEND
    # ========================================================

    source_label = (
        "WOA23 monthly climatology"
        if mode == "monthly"
        else "WOA23 annual climatology"
    )

    # Title placed at the top
    fig.suptitle(
        f"{SERIES_VARIABLES[variable]} | "
        f"{zmin:.0f}–{zmax:.0f} m\n"
        f"{source_label} (before 1993) + "
        f"CMEMS monthly (1993 onward)",
        fontsize=21,
        fontweight="bold",
        y=0.98
    )

    # Separate legend
    legend_handles = [
        Line2D(
            [0], [0],
            color=WOA_COLOR,
            lw=2.5,
            label=source_label
        ),
        Line2D(
            [0], [0],
            color=CMEMS_COLOR,
            lw=2.5,
            label="CMEMS monthly"
        )
    ]

    fig.legend(
        handles=legend_handles,
        loc="center",
        bbox_to_anchor=(0.5, 0.855),
        ncol=2,
        fontsize=15,
        frameon=False
    )

    # ========================================================
    # SAVE FIGURE TO MEMORY
    # ========================================================

    buffer = BytesIO()

    try:

        fig.savefig(
            buffer,
            format="png",
            dpi=150,
            bbox_inches="tight",
            facecolor="white"
        )

        return buffer.getvalue()

    finally:

        buffer.close()
        plt.close(fig)


# ============================================================
# 7. INTERACTIVE CONTROLS
# ============================================================

series_variable_selector = widgets.Dropdown(
    options=[
        (name, var)
        for var, name in SERIES_VARIABLES.items()
    ],
    value="no3",
    description="Variable:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px")
)

series_zmin_selector = widgets.BoundedFloatText(
    value=0.0,
    min=0.0,
    max=5500.0,
    step=50.0,
    description="Min depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

series_zmax_selector = widgets.BoundedFloatText(
    value=100.0,
    min=0.0,
    max=5500.0,
    step=50.0,
    description="Max depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

date_options = pd.date_range(
    f"{FIRST_YEAR}-01-01",
    f"{LAST_YEAR}-12-01",
    freq="MS"
)

date_choices = [
    (d.strftime("%Y-%m"), d)
    for d in date_options
]

series_start_selector = widgets.Dropdown(
    options=date_choices,
    value=pd.Timestamp("1960-01-01"),
    description="Start:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="190px")
)

series_end_selector = widgets.Dropdown(
    options=date_choices,
    value=pd.Timestamp(f"{LAST_YEAR}-12-01"),
    description="End:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="190px")
)

series_button = widgets.Button(
    description="Generate time series",
    button_style="primary",
    icon="line-chart",
    layout=widgets.Layout(width="220px")
)

series_status = widgets.HTML(
    value="<i>Select variable, depth range and dates.</i>"
)

series_image = widgets.Image(
    format="png"
)


# ============================================================
# 8. DYNAMIC DEPTH INFORMATION
# ============================================================

depth_info = widgets.HTML()


def update_depth_info(change=None):

    variable = series_variable_selector.value

    limit = MONTHLY_LIMITS[variable]

    depth_info.value = (
        f"<b>WOA23 depth ranges:</b> "
        f"Monthly: 0–{limit:.0f} m | "
        f"Annual: {limit:.0f}–5500 m. "
        f"Ranges crossing {limit:.0f} m are not allowed."
    )


series_variable_selector.observe(
    update_depth_info,
    names="value"
)

update_depth_info()


# ============================================================
# 9. BUTTON CALLBACK
# ============================================================

def on_series_clicked(button):

    series_button.disabled = True

    try:

        variable = series_variable_selector.value

        zmin = float(series_zmin_selector.value)
        zmax = float(series_zmax_selector.value)

        start_date = pd.Timestamp(
            series_start_selector.value
        )

        end_date = pd.Timestamp(
            series_end_selector.value
        )

        if start_date > end_date:
            raise ValueError(
                "Start date must be earlier than end date."
            )

        mode = validate_series_depth(
            variable,
            zmin,
            zmax
        )

        series_status.value = (
            "<b>Calculating time series...</b>"
        )

        image_data = plot_combined_series(
            variable,
            zmin,
            zmax,
            start_date,
            end_date
        )

        series_image.value = image_data

        series_status.value = (
            f"<b>Completed.</b> "
            f"WOA23 mode: {mode}. "
            f"Depth: {zmin:.0f}–{zmax:.0f} m. "
            f"Period: {start_date:%Y-%m} to "
            f"{end_date:%Y-%m}."
        )

    except Exception as exc:

        series_status.value = (
            f"<b style='color:red'>Error:</b> {exc}"
        )

        raise

    finally:

        series_button.disabled = False


series_button.on_click(on_series_clicked)


# ============================================================
# 10. DISPLAY INTERACTIVE PANEL
# ============================================================

controls1 = widgets.HBox([
    guided_control(series_variable_selector, 'Choose the biogeochemical variable to plot.'),
    guided_control(series_zmin_selector, 'Upper boundary of the layer to average (m).'),
    guided_control(series_zmax_selector, 'Lower boundary of the layer to average (m); must exceed Min depth.')
], layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

controls2 = widgets.HBox([
    guided_control(series_start_selector, 'First year and month to include (YYYY-MM).'),
    guided_control(series_end_selector, 'Last year and month to include (YYYY-MM).'),
    guided_control(series_button, 'Build time series for the selected boundaries only.')
], layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

display(
    widgets.HTML(
        "<h3>WOA23 + CMEMS — Combined monthly time series</h3>"
    ),
    widgets.HTML('<div style="padding:9px 12px;margin:4px 0 12px;background:#f2f6fb;border-left:3px solid #557ca3;color:#253a4b;"><b>Data sources:</b> Before 1993, WOA23 climatology is repeated each year (monthly in the upper ocean; annual at depth). From 1993, actual CMEMS monthly means are used. The shaded envelopes represent spatial and vertical standard deviations.</div>'),
    controls1,
    depth_info,
    controls2,
    series_status,
    series_image
)

## 10 — Interactive time–depth Hovmöllers (WOA23 + CMEMS)

The same geometry-aware masks are used for all boundaries. WOA23 monthly shallow / annual deep, CMEMS monthly since 1993.

In [ ]:
# ============================================================
# SECTION 10 — INTERACTIVE TIME-DEPTH HOVMOLLER DIAGRAMS
#
# WOA23 before 1993:
#   Monthly climatology at shallow depths
#   Annual climatology at deeper depths
#
# CMEMS from 1993 onward:
#   Original monthly fields
#
# Four +/-1 degree boundaries
# ============================================================

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets

from io import BytesIO
from IPython.display import display


# ============================================================
# 1. CONFIGURATION
# ============================================================

HOV_VARIABLES = {
    "no3": "Nitrate",
    "po4": "Phosphate",
    "si": "Silicate",
    "o2": "Dissolved oxygen"
}

# Maximum depth of WOA23 monthly climatologies
HOV_MONTHLY_LIMITS = {
    "no3": 800.0,
    "po4": 800.0,
    "si": 800.0,
    "o2": 1500.0
}

HOV_ANNUAL_MAX_DEPTH = 5500.0

HOV_FIRST_YEAR = 1960

HOV_LAST_YEAR = max(
    2026,
    int(ds_monthly.time.dt.year.max().values)
)

HOV_CMAP = "turbo"
HOV_MONTHS = np.arange(1, 13)


# ============================================================
# 2. VALIDATE USER SELECTION
# ============================================================

def validate_hov_selection(variable, start_year, end_year,
                           zmin, zmax):

    if variable not in HOV_VARIABLES:
        raise ValueError(
            f"Unknown variable: {variable}"
        )

    if start_year > end_year:
        raise ValueError(
            "Start year must be earlier than or equal to end year."
        )

    if zmin < 0 or zmax <= zmin:
        raise ValueError(
            "Depth range must satisfy 0 <= min < max."
        )

    if zmax > HOV_ANNUAL_MAX_DEPTH:
        raise ValueError(
            "Maximum depth is 5500 m for WOA23."
        )


# ============================================================
# 3. CMEMS TIME-DEPTH FIELD
# ============================================================

def cmems_hov_field(variable, bounds, start_year, end_year,
                    target_depths):
    """
    Calculate spatial mean within the boundary.

    Returns:
        time x depth

    Only available CMEMS months are used.
    """

    da = region_view(
        ds_monthly[variable],
        bounds
    )

    # Select requested period
    da = da.sel(
        time=slice(
            f"{max(1993, start_year)}-01-01",
            f"{end_year}-12-31"
        )
    )

    # Select target depth levels
    da = da.sel(
        depth=target_depths
    )

    # Spatial mean within boundary
    mean, _ = spatial_stats(da)

    return mean.compute()


# ============================================================
# 4. WOA23 MONTHLY AND ANNUAL PROFILES
# ============================================================

def woa_hov_profiles(variable, bounds, target_depths):
    """
    Calculate spatially averaged WOA23 profiles.

    Monthly climatology:
        12 months x depth

    Annual climatology:
        depth

    Interpolate to CMEMS depth levels without extrapolation.

    Below the monthly climatology depth limit,
    use the annual climatology.
    """

    limit = HOV_MONTHLY_LIMITS[variable]

    target_depths = np.asarray(
        target_depths,
        dtype=float
    )

    # --------------------------------------------------------
    # Monthly WOA23
    # --------------------------------------------------------

    wo_monthly = region_view(
        woa_clim[variable],
        bounds
    )

    # Spatial mean
    monthly_mean, _ = spatial_stats(
        wo_monthly
    )

    monthly_mean = monthly_mean.reindex(
        month=HOV_MONTHS
    )

    # Do not include depths beyond monthly coverage
    monthly_mean = monthly_mean.sel(
        depth=slice(None, limit)
    )

    # Interpolate to CMEMS depths
    monthly_interp = monthly_mean.interp(
        depth=target_depths,
        method="linear",
        kwargs={"fill_value": np.nan}
    )

    # --------------------------------------------------------
    # Annual WOA23
    # --------------------------------------------------------

    if woa_annual is None:
        raise FileNotFoundError(
            "WOA_ANNUAL.nc is not loaded."
        )

    wo_annual = region_view(
        woa_annual[variable],
        bounds
    )

    annual_mean, _ = spatial_stats(
        wo_annual
    )

    # Remove singleton dimensions if present
    annual_mean = annual_mean.squeeze(
        drop=True
    )

    annual_interp = annual_mean.interp(
        depth=target_depths,
        method="linear",
        kwargs={"fill_value": np.nan}
    )

    # --------------------------------------------------------
    # Combine monthly and annual climatologies
    # --------------------------------------------------------

    # Repeat annual climatology for all 12 months
    annual_12 = annual_interp.expand_dims(
        month=HOV_MONTHS
    )

    annual_12 = annual_12.transpose(
        "month",
        "depth"
    )

    monthly_interp = monthly_interp.transpose(
        "month",
        "depth"
    )

    # Monthly values above the depth limit.
    # Annual values below it.
    #
    # At shallow depths where WOA monthly values
    # are missing, annual WOA values are used.

    combined = xr.where(
        (monthly_interp.depth <= limit)
        & monthly_interp.notnull(),
        monthly_interp,
        annual_12
    )

    return combined.compute()


# ============================================================
# 5. BUILD COMPLETE TIME-DEPTH FIELD
# ============================================================

def build_hov_field(variable, bounds, start_year, end_year,
                    zmin, zmax):
    """
    Build time x depth field combining:

        WOA23 before 1993
        CMEMS from 1993 onward

    WOA23 climatology is repeated each year.
    Missing CMEMS months remain NaN.
    """

    # --------------------------------------------------------
    # Target depth grid
    # --------------------------------------------------------

    depth_all = np.asarray(
        ds_monthly.depth.values,
        dtype=float
    )

    target_depths = depth_all[
        (depth_all >= zmin)
        & (depth_all <= zmax)
    ]

    if len(target_depths) == 0:
        raise ValueError(
            "No CMEMS depth levels inside the selected range."
        )

    # --------------------------------------------------------
    # Complete monthly time axis
    # --------------------------------------------------------

    all_time = pd.date_range(
        f"{start_year}-01-01",
        f"{end_year}-12-01",
        freq="MS"
    )

    # Initialize with NaN
    values = np.full(
        (len(all_time), len(target_depths)),
        np.nan,
        dtype=float
    )

    # --------------------------------------------------------
    # WOA23 BEFORE 1993
    # --------------------------------------------------------

    before_mask = all_time.year < 1993

    if np.any(before_mask):

        wo = woa_hov_profiles(
            variable,
            bounds,
            target_depths
        )

        # WOA climatology: month x depth
        wo_values = np.asarray(
            wo.transpose("month", "depth").values
        )

        # Select corresponding month for each year
        month_indices = (
            all_time[before_mask].month.values - 1
        )

        values[before_mask, :] = wo_values[
            month_indices, :
        ]

    # --------------------------------------------------------
    # CMEMS FROM 1993
    # --------------------------------------------------------

    after_mask = all_time.year >= 1993

    if np.any(after_mask):

        cm = cmems_hov_field(
            variable,
            bounds,
            start_year,
            end_year,
            target_depths
        )

        # Align available CMEMS records with complete calendar
        cm = cm.assign_coords(
            time=pd.to_datetime(cm.time.values)
        )

        cm = cm.reindex(
            time=all_time[after_mask]
        )

        values[after_mask, :] = np.asarray(
            cm.transpose("time", "depth").values
        )

    # --------------------------------------------------------
    # Final DataArray
    # --------------------------------------------------------

    combined = xr.DataArray(
        values,
        dims=("time", "depth"),
        coords={
            "time": all_time,
            "depth": target_depths
        },
        name=variable
    )

    return combined


# ============================================================
# 6. PLOTTING FUNCTION
# ============================================================

def plot_interactive_hov(variable, start_year, end_year,
                         zmin, zmax):

    validate_hov_selection(
        variable,
        start_year,
        end_year,
        zmin,
        zmax
    )

    # --------------------------------------------------------
    # Calculate all boundaries
    # --------------------------------------------------------

    fields = {}

    for name, bounds in REGIONS.items():

        fields[name] = build_hov_field(
            variable,
            bounds,
            start_year,
            end_year,
            zmin,
            zmax
        )

    # --------------------------------------------------------
    # Shared color limits
    # --------------------------------------------------------

    valid_values = []

    for da in fields.values():

        values = np.asarray(da.values)

        valid = values[np.isfinite(values)]

        if valid.size:
            valid_values.append(valid)

    if valid_values:

        combined_values = np.concatenate(valid_values)

        vmin, vmax = np.nanpercentile(
            combined_values,
            [2, 98]
        )

    else:

        vmin, vmax = 0.0, 1.0

    if vmax <= vmin:
        vmax = vmin + 1.0

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    cols=min(2,len(REGIONS)); rows=int(np.ceil(len(REGIONS)/cols))
    fig, axes=plt.subplots(rows,cols,figsize=(9.5*cols,6*rows),
                           sharex=True,sharey=True,constrained_layout=False,squeeze=False)
    for unused in axes.flat[len(REGIONS):]:unused.set_visible(False)

    fig.subplots_adjust(
        top=0.84,
        bottom=0.10,
        left=0.08,
        right=0.88,
        hspace=0.30,
        wspace=0.22
    )

    # --------------------------------------------------------
    # Selected boundary panels
    # --------------------------------------------------------

    last_mesh = None

    for ax, (name, data) in zip(
        axes.flat,
        fields.items()
    ):

        # Hovmoller
        last_mesh = data.transpose(
            "depth",
            "time"
        ).plot.pcolormesh(
            ax=ax,
            x="time",
            y="depth",
            cmap=HOV_CMAP,
            vmin=vmin,
            vmax=vmax,
            add_colorbar=False
        )

        # ----------------------------------------------------
        # Transition between WOA23 and CMEMS
        # ----------------------------------------------------

        transition = pd.Timestamp("1993-01-01")

        if start_year < 1993 <= end_year:

            ax.axvline(
                transition,
                color="white",
                linestyle="--",
                linewidth=2
            )

        # ----------------------------------------------------
        # Monthly/annual climatology transition depth
        # ----------------------------------------------------

        depth_limit = HOV_MONTHLY_LIMITS[variable]

        if zmin < depth_limit < zmax and start_year < 1993:

            # Draw only over the WOA23 period
            x_start = pd.Timestamp(
                f"{start_year}-01-01"
            )

            x_end = pd.Timestamp(
                f"{min(end_year, 1992)}-12-31"
            )

            ax.hlines(
                depth_limit,
                xmin=x_start,
                xmax=x_end,
                colors="white",
                linestyles=":",
                linewidth=1.8
            )

        # ----------------------------------------------------
        # Axes
        # ----------------------------------------------------

        ax.set_title(
            f"{name} boundary",
            fontsize=19,
            fontweight="bold",
            pad=12
        )

        ax.set_ylabel(
            "Depth (m)",
            fontsize=16
        )

        ax.set_xlabel(
            "Year",
            fontsize=16
        )

        # Surface at top
        ax.set_ylim(zmax, zmin)

        ax.tick_params(
            axis="both",
            labelsize=14
        )

        ax.grid(
            alpha=0.15,
            linestyle=":"
        )

        # Date formatting
        ax.xaxis.set_major_locator(
            mdates.AutoDateLocator(
                minticks=4,
                maxticks=8
            )
        )

        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%Y")
        )

    # --------------------------------------------------------
    # Shared colorbar
    # --------------------------------------------------------

    cbar_ax = fig.add_axes([
        0.905,
        0.15,
        0.018,
        0.62
    ])

    cbar = fig.colorbar(
        last_mesh,
        cax=cbar_ax
    )

    cbar.set_label(
        "Concentration (mmol m$^{-3}$)",
        fontsize=16
    )

    cbar.ax.tick_params(
        labelsize=14
    )

    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        f"{HOV_VARIABLES[variable]} | "
        f"{zmin:.0f}–{zmax:.0f} m | "
        f"{start_year}–{end_year}\n"
        f"WOA23 climatology (before 1993) + "
        f"CMEMS monthly (1993 onward)",
        fontsize=21,
        fontweight="bold",
        y=0.97
    )

    # --------------------------------------------------------
    # Save to PNG for interactive display
    # --------------------------------------------------------

    buffer = BytesIO()

    try:

        fig.savefig(
            buffer,
            format="png",
            dpi=150,
            bbox_inches="tight",
            facecolor="white"
        )

        return buffer.getvalue()

    finally:

        buffer.close()
        plt.close(fig)


# ============================================================
# 7. INTERACTIVE CONTROLS
# ============================================================

# Variable
hov_variable_selector = widgets.Dropdown(
    options=[
        (label, var)
        for var, label in HOV_VARIABLES.items()
    ],
    value="no3",
    description="Variable:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px")
)

# Starting year
hov_start_selector = widgets.BoundedIntText(
    value=1960,
    min=1960,
    max=HOV_LAST_YEAR,
    description="Start year:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="200px")
)

# Ending year
hov_end_selector = widgets.BoundedIntText(
    value=HOV_LAST_YEAR,
    min=1960,
    max=HOV_LAST_YEAR,
    description="End year:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="190px")
)

# Minimum depth
hov_zmin_selector = widgets.BoundedFloatText(
    value=0.0,
    min=0.0,
    max=5500.0,
    step=50.0,
    description="Min depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

# Maximum depth
hov_zmax_selector = widgets.BoundedFloatText(
    value=800.0,
    min=0.0,
    max=5500.0,
    step=50.0,
    description="Max depth:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="190px")
)

# Button
hov_button = widgets.Button(
    description="Generate Hovmöller",
    button_style="primary",
    icon="area-chart",
    layout=widgets.Layout(width="220px")
)

# Status
hov_status = widgets.HTML(
    value="<i>Select variable, dates and depth range.</i>"
)

# Figure output
hov_image = widgets.Image(
    format="png"
)


# ============================================================
# 8. DYNAMIC INFORMATION
# ============================================================

hov_depth_info = widgets.HTML()


def update_hov_depth_info(change=None):

    variable = hov_variable_selector.value

    limit = HOV_MONTHLY_LIMITS[variable]

    hov_depth_info.value = (
        f"<b>WOA23:</b> "
        f"Monthly climatology: 0–{limit:.0f} m | "
        f"Annual climatology: below {limit:.0f} m. "
        f"Depth ranges crossing this limit are allowed."
    )


hov_variable_selector.observe(
    update_hov_depth_info,
    names="value"
)

update_hov_depth_info()


# ============================================================
# 9. BUTTON CALLBACK
# ============================================================

def on_hov_clicked(button):

    hov_button.disabled = True

    try:

        variable = hov_variable_selector.value

        start_year = int(
            hov_start_selector.value
        )

        end_year = int(
            hov_end_selector.value
        )

        zmin = float(
            hov_zmin_selector.value
        )

        zmax = float(
            hov_zmax_selector.value
        )

        hov_status.value = (
            "<b>Calculating Hovmöller diagrams...</b>"
        )

        image_data = plot_interactive_hov(
            variable,
            start_year,
            end_year,
            zmin,
            zmax
        )

        hov_image.value = image_data

        hov_status.value = (
            f"<b>Completed:</b> "
            f"{HOV_VARIABLES[variable]} | "
            f"{start_year}–{end_year} | "
            f"{zmin:.0f}–{zmax:.0f} m."
        )

    except Exception as exc:

        hov_status.value = (
            f"<b style='color:red'>Error:</b> {exc}"
        )

        raise

    finally:

        hov_button.disabled = False


hov_button.on_click(
    on_hov_clicked
)


# ============================================================
# 10. DISPLAY INTERACTIVE PANEL
# ============================================================

controls1 = widgets.HBox([
    guided_control(hov_variable_selector, 'Choose nitrate, phosphate, silicate or dissolved oxygen.'),
    guided_control(hov_start_selector, 'First calendar year of the Hovmöller diagram.'),
    guided_control(hov_end_selector, 'Last calendar year (must be at least the start year).')
], layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

controls2 = widgets.HBox([
    guided_control(hov_zmin_selector, 'Top of the displayed depth range (m; 0 = surface).'),
    guided_control(hov_zmax_selector, 'Bottom of the displayed depth range (m), up to 5500 m.'),
    guided_control(hov_button, 'Generate time–depth plots for all selected ocean boundaries.')
], layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

display(
    widgets.HTML(
        "<h3>WOA23 + CMEMS — Interactive time-depth Hovmöller</h3>"
    ),
    widgets.HTML('<div style="padding:9px 12px;margin:4px 0 12px;background:#f2f6fb;border-left:3px solid #557ca3;color:#253a4b;"><b>How to read:</b> Horizontal axis = year; vertical axis = depth (0 m at the top). Before 1993 the repeated WOA23 monthly climatology is used where available, with annual WOA23 below its depth coverage; from 1993 onward CMEMS monthly fields are shown.</div>'),
    controls1,
    controls2,
    hov_depth_info,
    hov_status,
    hov_image
)

## Reproducibility and contributor notes

- The downloaders are unchanged: CMEMS/WOA23 are downloaded on a **geographic rectangle**. The visualization domain can be the same rectangle or the perimeter of a **NetCDF model grid**.
- In Section 1 choose **Yes — use downloaded area** or **No — read boundaries from .nc**. There is no manual rotation mode. If **Yes**, the extents are inferred from **loaded CMEMS coordinates in Section 2**; if **No**, load a `.nc` using the file path or upload widget.
- A model grid should contain matching 2-D `lon_rho`/`lat_rho` (or supported equivalents). The first/last rows and columns define the model perimeter. Verify ghost/halo cells and which sides are **open**. Select those in Section 1.
- Band half-width can be specified as degrees or kilometres. **Default ±1°** preserves the earlier coordinate-based sampling near rectangular boundaries. For geographically consistent distances choose kilometres. Polygon bands can overlap at corners; each boundary has its own statistics.
- In Section 3 the **solid black** perimeter is the actual visualization/model domain; the **gray dashed** rectangle is the CMEMS data envelope. Grid points represent original product positions, not model-grid nodes.
- The displayed downloaded extent is taken from coordinate centers in the loaded CMEMS subset and may differ slightly from user-requested API limits (by up to a grid spacing). Small tolerance warnings are not evidence of a data error.
- For the NetCDF option, the notebook reads boundary coordinates; it does **not** automatically determine where ocean-open boundaries end or where land boundaries begin. Choose active edges and inspect the map.
- The workflow requires Python with xarray, numpy, scipy, pandas, matplotlib, dask, ipywidgets, shapely >= 2, pyproj, and cartopy. Run from inside the `codeblue-obc-ini` repository. Real datasets may require significant RAM; CMEMS files are read lazily when possible.
- Do not commit raw NetCDF downloads or private access credentials to the public GitHub repository. Keep uploaded grids in `data/` and note their original source for reproducibility.

- **Active/open boundaries:** Section 1 uses checkboxes for North, South, West, and East. Unticked boundaries are excluded from masks and all boundary-specific figures. CodeBlue defaults to North/South/West; adjust this for your model and click **Apply geometry**. Re-execute downstream cells after a change because notebook results are not automatically refreshed.


- **Avoid using downloaded coordinates as model boundaries.** When selecting Manual model bounds, enter the original, unpadded limits. The coverage of downloaded NetCDFs is reported separately after loading; do not infer study-domain edges from the grid-data envelope.

- **Mediterranean exclusion:** On by default in Section 1. A documented approximate Mediterranean polygon masks (NaN) the relevant CMEMS and WOA23 grid points in memory, including their map grid-point markers. This is intended for the NE Atlantic CodeBlue setting; for other geographic regions, inspect or adjust the polygon. After changing the checkbox, rerun Section 2 and all subsequent analysis cells.
